# BUPA DDEFI - Conditions + MedicationRequest (version simplifiée)

In [1]:
# Run once to reset everything and remove already filtered data
from pathlib import Path
import shutil

filtered_dir = Path("synthea_local_simplified/filtered_conditions_medreq")
shutil.rmtree(filtered_dir, ignore_errors=True)

## 0) Installation (Colab)

In [2]:
# Run once on Colab
!pip -q install "pandas>=2.0" "matplotlib>=3.7" "requests>=2.31" "pydantic>=2.0" "tqdm>=4.66" >/dev/null
!pip -q install "transformers>=4.41" "accelerate>=0.30" "sentencepiece>=0.2" "huggingface_hub>=0.23" >/dev/null
!pip -q install "bitsandbytes>=0.43" >/dev/null || true


## 0.1) Authentification Hugging Face (si MedGemma est gated)

Ajoute un secret Colab `HF_TOKEN` (token read), puis exécute la cellule.


In [3]:
import os
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or userdata.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face: authenticated.")
else:
    print("No HF_TOKEN found. Add a Colab secret named HF_TOKEN if the model is gated.")


Hugging Face: authenticated.


## 1) Imports, chemins, configuration

In [4]:
from __future__ import annotations

import json
import os
import random
import re
import shutil
import zipfile
import warnings
from datetime import datetime, timezone
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import requests
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pydantic import BaseModel, Field
from IPython.display import display, JSON

# Paths
DATA_ROOT = Path("synthea_local_simplified")
ZIP_DIR = DATA_ROOT / "zip"
RAW_DIR = DATA_ROOT / "extracted_raw"
FILTERED_DIR = DATA_ROOT / "filtered_conditions_medreq"  # 1 JSON Bundle per patient (Patient + Condition + MedicationRequest)

PIPE_DIR = Path("data_bupa_conditions_medreq")
GOLD_DIR = PIPE_DIR / "gold"
NOTES_DIR = PIPE_DIR / "notes"
RAW_NOTES_DIR = NOTES_DIR / "raw"
RECON_DIR = PIPE_DIR / "reconstruction"
SCORES_DIR = PIPE_DIR / "scores"
FILTER_CACHE_DIR = PIPE_DIR / "filter_cache"  # hybrid filtering cache (patient-level)
VERIFICATION_DIR = PIPE_DIR / "verification"

for d in [ZIP_DIR, RAW_DIR, FILTERED_DIR, GOLD_DIR, NOTES_DIR, RAW_NOTES_DIR, RECON_DIR, SCORES_DIR, FILTER_CACHE_DIR, VERIFICATION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =========================
# Config
# =========================
MODEL_NAME = "google/medgemma-4b-it"
N_PATIENTS = 40
RANDOM_SEED = 150

# Hybrid filtering
# Step A: deterministic prefilter (conservative) to reduce initial size (fast, reproducible).
PREFILTER_ENABLED = True
PREFILTER_DROP_SOCIAL_HISTORY = True

# Only keep coded conditions in these systems (stable, helps remove junk)
ALLOWED_CONDITION_SYSTEMS = {
    "http://snomed.info/sct",
    "http://hl7.org/fhir/sid/icd-10",
    "http://hl7.org/fhir/sid/icd-10-cm",
}

# Very conservative "obvious SDOH/admin" substrings for prefilter.
# Keep this short to avoid dropping true medical conditions before the LLM sees them.
PREFILTER_DROP_SUBSTRINGS = [
    "criminal record",
    "victim of",
    "intimate partner",
    "domestic violence",
    "violence in the environment",
    "reports of violence",
    "housing unsatisfactory",
    "not in labor force",
    "employment",
    "education",
    "social",
    "stress",
    "risk activity",
    "primary school",
]

# Candidate caps BEFORE LLM filtering (keep higher than final caps)
PREFILTER_MAX_CONDITION_CANDIDATES = 80
PREFILTER_MAX_MEDREQ_CANDIDATES = 40

# Step B: LLM patient-level filter on the selected sample only (high precision).
USE_LLM_PATIENT_FILTER = True
LLM_FILTER_MAX_NEW_TOKENS = 512
LLM_FILTER_TEMPERATURE = 0.0
LLM_FILTER_DO_SAMPLE = False
LLM_FILTER_RETRIES = 1

# Final caps used for downstream note + reconstruction + scoring
MAX_CONDITIONS_PER_PATIENT = 12
MAX_MEDICATION_REQUESTS_PER_PATIENT = 12

PREVIEW_N = 6



# Centralized generation budgets
NOTE_GENERATION_MAX_NEW_TOKENS = 320
RECON_EXTRACTION_MAX_NEW_TOKENS = 220
RECON_FIX_JSON_MAX_NEW_TOKENS = 256
SEMANTIC_MATCH_MAX_NEW_TOKENS = 96
NOTE_CONSISTENCY_MAX_NEW_TOKENS = 256
VERIFICATION_MAX_NEW_TOKENS = 256

# Centralized temperatures / sampling
NOTE_GENERATION_TEMPERATURE = 0.8
NOTE_GENERATION_DO_SAMPLE = True
RECON_TEMPERATURE = 0.0
RECON_DO_SAMPLE = False
SCORING_TEMPERATURE = 0.0
SCORING_DO_SAMPLE = False
VERIFICATION_TEMPERATURE = 0.0
VERIFICATION_DO_SAMPLE = False

# Reduce noisy transformers warnings about max_length when max_new_tokens is already controlled.
warnings.filterwarnings("ignore", message=r"Both `max_new_tokens`.*`max_length`.*")

random.seed(RANDOM_SEED)

print("DATA_ROOT:", DATA_ROOT.resolve())
print("PIPE_DIR:", PIPE_DIR.resolve())


DATA_ROOT: /content/synthea_local_simplified
PIPE_DIR: /content/data_bupa_conditions_medreq


## Guide de lecture du notebook

### Logique métier du pipeline

Ce notebook suit la chaîne logique suivante :

1. **Préparer l’environnement**
2. **Télécharger et extraire les Bundles FHIR**
3. **Préfiltrer les données**
4. **Construire la source filtrée de référence** qui servira de vérité de départ
5. **Générer des notes non structurées**
6. **Reconstruire une structure depuis les notes**
7. **Scorer la reconstruction**
8. **Analyser les erreurs**
9. **Relire les fichiers produits**
10. **Exporter les datasets consolidés**

### Nomenclature canonique

Pour éviter les ambiguïtés, les objets doivent se lire ainsi :

- `gold` ou `gold_source` : source structurée filtrée de référence
- `raw_note_text` : sortie brute du LLM de génération
- `note_text` : note nettoyée utilisée pour la reconstruction
- `recon_raw` : extraction brute renvoyée par le LLM
- `recon_clean` : reconstruction principale utilisée pour les scores
- `recon_clipped` : reconstruction stricte debug par overlap exact
- `df_scores` : table des scores
- `df_error_taxonomy` : table d’interprétation des erreurs

### Important

L’ordre **logique** du pipeline et l’ordre **technique** des cellules ne sont pas toujours exactement les mêmes.  
Par exemple, le modèle est chargé à un endroit qui sert ensuite à plusieurs étapes, même si, du point de vue métier, certaines opérations appartiennent encore à la construction du `gold_source`.

## 2) Télécharger et extraire Synthea (local)

In [5]:
SYNTHEA_ZIP_URLS = [
    "https://synthetichealth.github.io/synthea-sample-data/downloads/synthea_sample_data_fhir_r4_nov2021.zip",
    "https://github.com/synthetichealth/synthea-sample-data/raw/refs/heads/main/downloads/synthea_sample_data_fhir_r4_nov2021.zip",
]
ZIP_PATH = ZIP_DIR / "synthea_fhir_r4.zip"

def download_file(url: str, dest: Path, timeout_s: int = 60) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=timeout_s) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        pbar = tqdm(total=total, unit="B", unit_scale=True, desc=f"Downloading {dest.name}")
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
        pbar.close()

def ensure_zip(zip_path: Path) -> Path:
    if zip_path.exists() and zip_path.stat().st_size > 0:
        print("Zip already present:", zip_path.resolve())
        return zip_path
    last_err = None
    for url in SYNTHEA_ZIP_URLS:
        try:
            print("Trying:", url)
            download_file(url, zip_path)
            return zip_path
        except Exception as e:
            last_err = e
            if zip_path.exists():
                try: zip_path.unlink()
                except Exception: pass
    raise RuntimeError(f"Could not download zip. Last error: {last_err}")

def extract_zip(zip_path: Path, extract_dir: Path) -> None:
    marker = extract_dir / ".extracted_ok"
    if marker.exists():
        print("Already extracted:", extract_dir.resolve())
        return
    if extract_dir.exists():
        for p in extract_dir.iterdir():
            if p.is_file():
                p.unlink()
            else:
                shutil.rmtree(p)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)
    marker.write_text("ok", encoding="utf-8")
    print("Extracted to:", extract_dir.resolve())

zip_path = ensure_zip(ZIP_PATH)
extract_zip(zip_path, RAW_DIR)


Trying: https://synthetichealth.github.io/synthea-sample-data/downloads/synthea_sample_data_fhir_r4_nov2021.zip


Extracted to: /content/synthea_local_simplified/extracted_raw


## 3) Filtrer en amont (un seul passage) et créer un dataset dérivé Conditions + MedicationRequest

### 3A) Fonctions utilitaires et règles de préfiltrage

In [6]:
def safe_read_json(path: Path) -> Optional[dict]:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def first(lst: List[Any], default=None):
    return lst[0] if lst else default

def norm_text(s: Optional[str]) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def find_patient(bundle: dict) -> Optional[dict]:
    for e in bundle.get("entry", []) or []:
        r = e.get("resource", {})
        if r.get("resourceType") == "Patient":
            return r
    return None

def iter_resources(bundle: dict, resource_type: str) -> List[dict]:
    out = []
    for e in bundle.get("entry", []) or []:
        r = e.get("resource", {})
        if r.get("resourceType") == resource_type:
            out.append(r)
    return out

def cond_display(cond: dict) -> Optional[str]:
    code = cond.get("code", {}) or {}
    coding = first(code.get("coding", []), {}) or {}
    return coding.get("display") or code.get("text")

def cond_system_code(cond: dict) -> Tuple[str, str]:
    code = cond.get("code", {}) or {}
    coding = first(code.get("coding", []), {}) or {}
    return (coding.get("system") or "", coding.get("code") or "")

def is_social_history(cond: dict) -> bool:
    # Synthea often uses Condition to store SDOH. Category is the most reliable deterministic signal when present.
    for cat in cond.get("category", []) or []:
        for coding in (cat.get("coding", []) or []):
            code = norm_text(coding.get("code"))
            disp = norm_text(coding.get("display"))
            if code == "social-history":
                return True
            # be permissive: any explicit social / sdoh category is treated as noise
            if "social" in code or "social" in disp or "sdoh" in code or "sdoh" in disp:
                return True
    return False

_NOISY_REGEX_COMPILED = None

def is_noisy_display(disp: Optional[str]) -> bool:
    """Conservative prefilter: drop only obvious non-medical/SDoH patterns."""
    if not PREFILTER_ENABLED:
        return False
    d = norm_text(disp)
    if not d:
        return False
    for sub in (PREFILTER_DROP_SUBSTRINGS or []):
        subn = norm_text(sub)
        if subn and subn in d:
            return True
    return False

def is_medical_condition(cond: dict) -> bool:
    """Step A prefilter rule (conservative): keep coded medical conditions, drop obvious SDoH."""
    if PREFILTER_DROP_SOCIAL_HISTORY and is_social_history(cond):
        return False

    disp = cond_display(cond)
    if is_noisy_display(disp):
        return False

    system, code = cond_system_code(cond)
    if not system or not code:
        return False
    if system not in ALLOWED_CONDITION_SYSTEMS:
        return False
    return True

def dedupe_limit_conditions(conds: List[dict], max_items: int) -> List[dict]:
    out = []
    seen = set()
    for c in conds:
        system, code = cond_system_code(c)
        key = f"{norm_text(system)}|{norm_text(code)}"
        if not system or not code:
            continue
        if key in seen:
            continue
        seen.add(key)
        out.append(c)
        if len(out) >= max_items:
            break
    return out



def med_display(medreq: dict) -> Optional[str]:
    med = medreq.get("medicationCodeableConcept", {}) or {}
    coding = first(med.get("coding", []), {}) or {}
    return coding.get("display") or med.get("text")

def med_system_code(medreq: dict) -> Tuple[str, str]:
    med = medreq.get("medicationCodeableConcept", {}) or {}
    coding = first(med.get("coding", []), {}) or {}
    return (coding.get("system") or "", coding.get("code") or "")

def dedupe_limit_medreqs(meds: List[dict], max_items: int) -> List[dict]:
    out = []
    seen = set()
    for m in meds:
        system, code = med_system_code(m)
        key = f"{norm_text(system)}|{norm_text(code)}"
        if not system or not code:
            # fallback to display
            disp = norm_text(med_display(m))
            if not disp:
                continue
            key = disp
        if key in seen:
            continue
        seen.add(key)
        out.append(m)
        if len(out) >= max_items:
            break
    return out

### 3B) Construction du dataset filtré (cache disque)

In [7]:
def build_filtered_dataset(raw_dir: Path, out_dir: Path) -> pd.DataFrame:
    marker = out_dir / ".filtered_ok"
    if marker.exists():
        print("Filtered dataset already present:", out_dir.resolve())
        print("Note: if you changed PREFILTER_* settings, delete FILTERED_DIR to rebuild.")
        # quick index
        rows = []
        for p in sorted(out_dir.glob("*.json")):
            pid = p.stem
            obj = safe_read_json(p)
            rows.append({
                "patient_id": pid,
                "file_path": str(p),
                "n_conditions": len(iter_resources(obj, "Condition")) if obj else 0,
                "n_medication_requests": len(iter_resources(obj, "MedicationRequest")) if obj else 0,
            })
        return pd.DataFrame(rows)

    rows = []
    json_files = sorted(raw_dir.rglob("*.json"))
    for p in tqdm(json_files, desc="Filtering Bundles (one pass)"):
        obj = safe_read_json(p)
        if not obj or obj.get("resourceType") != "Bundle":
            continue
        patient = find_patient(obj)
        if not patient:
            continue
        pid = patient.get("id")
        if not pid:
            continue

        conds = [c for c in iter_resources(obj, "Condition") if is_medical_condition(c)]
        conds = dedupe_limit_conditions(conds, max_items=PREFILTER_MAX_CONDITION_CANDIDATES)

        meds = iter_resources(obj, "MedicationRequest")
        meds = dedupe_limit_medreqs(meds, max_items=PREFILTER_MAX_MEDREQ_CANDIDATES)

        filtered = {
            "resourceType": "Bundle",
            "type": "collection",
            "entry": [{"resource": patient}] + [{"resource": c} for c in conds] + [{"resource": m} for m in meds],
        }
        out_path = out_dir / f"{pid}.json"
        out_path.write_text(json.dumps(filtered, ensure_ascii=False, indent=2), encoding="utf-8")

        rows.append({"patient_id": pid, "file_path": str(out_path), "n_conditions": len(conds), "n_medication_requests": len(meds)})

    marker.write_text("ok", encoding="utf-8")
    return pd.DataFrame(rows)

### 3C) Index des patients filtrés

In [8]:
df_index = build_filtered_dataset(RAW_DIR, FILTERED_DIR)
print("Filtered patients:", len(df_index))
display(df_index.head(10))


Filtering Bundles (one pass):   0%|          | 0/557 [00:00<?, ?it/s]

Filtered patients: 555


,patient_id,file_path,n_conditions,n_medication_requests
0,b0a06ead-cc42-aa48-dad6-841d4aa679fa,synthea_local_simplified/filtered_conditions_m...,9,7
1,ccfc4db2-2026-7adb-3db0-33f3828140bb,synthea_local_simplified/filtered_conditions_m...,8,4
2,31a2e8ec-69fc-8a71-3ab6-36cbdd508713,synthea_local_simplified/filtered_conditions_m...,9,1
3,71a8b156-760b-df6b-859e-eefc7932a526,synthea_local_simplified/filtered_conditions_m...,3,3
4,76b289fd-e825-734c-8446-316f59643593,synthea_local_simplified/filtered_conditions_m...,19,7
5,92fb7efc-5cfd-f8d3-927b-42f8ee099531,synthea_local_simplified/filtered_conditions_m...,9,5
6,81aa7647-779f-fd6b-94cf-782e606efeb2,synthea_local_simplified/filtered_conditions_m...,1,1
7,346a1435-2455-914f-c287-7b88052d05db,synthea_local_simplified/filtered_conditions_m...,8,8
8,1cfa5a70-7f3c-4227-5cf1-e182fcff4cd4,synthea_local_simplified/filtered_conditions_m...,15,4
9,97899f1d-9c3b-2b90-17b8-400c11ab8f0f,synthea_local_simplified/filtered_conditions_m...,9,8


## 4) Construction de la source filtrée de référence (`gold_source`)

Cette étape construit la vérité de départ du pipeline.  
Elle sélectionne un échantillon, extrait les champs utiles depuis les Bundles FHIR puis prépare la source qui servira à générer les notes.

### 4A) Extraire une représentation patient simple depuis le Bundle FHIR

In [9]:
# Inspired by the "extract_complete_patient_data" idea: one canonical extractor.

def load_bundle(path: str) -> dict:
    obj = safe_read_json(Path(path))
    if not obj:
        raise RuntimeError(f"Could not read: {path}")
    return obj

def patient_demographics(patient: dict) -> dict:
    name = first(patient.get("name", []), {}) or {}
    given = " ".join(name.get("given", []) or []).strip() or None
    family = (name.get("family") or "").strip() or None
    full_name = " ".join([x for x in [given, family] if x]).strip() or None
    return {
        "given": given,
        "family": family,
        "full_name": full_name,
        "birthDate": patient.get("birthDate"),
        "gender": patient.get("gender"),
    }

def extract_patient_record_prefilter(fhir_bundle: dict) -> dict:
    """Extract candidates from the prefiltered bundle. No LLM yet."""
    patient = find_patient(fhir_bundle) or {}
    pid = patient.get("id")
    demo = patient_demographics(patient)

    # Condition candidates (keep structured info to help the LLM decide)
    cond_candidates: List[dict] = []
    for c in iter_resources(fhir_bundle, "Condition"):
        disp = cond_display(c)
        if not disp:
            continue
        system, code = cond_system_code(c)
        cond_candidates.append({"system": system, "code": code, "display": disp})

    # Deduplicate by system|code preserving order
    seen = set()
    cond_candidates_dedup = []
    for it in cond_candidates:
        key = f'{norm_text(it.get("system"))}|{norm_text(it.get("code"))}'
        if not it.get("system") or not it.get("code"):
            continue
        if key in seen:
            continue
        seen.add(key)
        cond_candidates_dedup.append(it)
    cond_candidates = cond_candidates_dedup[:PREFILTER_MAX_CONDITION_CANDIDATES]

    # MedicationRequest candidates (we keep as-is, mostly already clean)
    med_candidates: List[str] = []
    for m in iter_resources(fhir_bundle, "MedicationRequest"):
        d = med_display(m)
        if d:
            med_candidates.append(d)
    med_candidates = list(dict.fromkeys(med_candidates))[:PREFILTER_MAX_MEDREQ_CANDIDATES]

    return {
        "patient_id": pid,
        **demo,
        "conditions_candidates": cond_candidates,
        "conditions": [x["display"] for x in cond_candidates],  # display list for preview (will be LLM-filtered later)
        "medication_requests": med_candidates,
    }

### 4B) Choisir l’échantillon et construire la source préfiltrée

On passe ici du Bundle FHIR filtré à une première source structurée lisible par le reste du pipeline.

In [10]:
# Choose sample
df_sample = df_index.sample(n=min(N_PATIENTS, len(df_index)), random_state=RANDOM_SEED).reset_index(drop=True)
bundles = [load_bundle(p) for p in df_sample["file_path"].tolist()]
patients_prefilter = [extract_patient_record_prefilter(b) for b in bundles]

df_gold_prefilter = pd.DataFrame([{
    "patient_id": p["patient_id"],
    "full_name": p.get("full_name"),
    "birthDate": p.get("birthDate"),
    "gender": p.get("gender"),
    "n_conditions_candidates": len(p["conditions"]),
    "conditions_candidates_preview": "; ".join(p["conditions"][:PREVIEW_N]),
    "n_medication_requests": len(p["medication_requests"]),
    "medreq_preview": "; ".join(p["medication_requests"][:PREVIEW_N]),
} for p in patients_prefilter]).sort_values("patient_id")

print("Prefiltered sample (candidates before LLM filtering)")
display(df_gold_prefilter)

# Show one example Bundle (prefiltered)
example_pid = df_gold_prefilter.iloc[0]["patient_id"]
display(JSON(load_bundle(str(FILTERED_DIR / f"{example_pid}.json"))))


Prefiltered sample (candidates before LLM filtering)


,patient_id,full_name,birthDate,gender,n_conditions_candidates,conditions_candidates_preview,n_medication_requests,medreq_preview
29,01f8bbfd-cfc6-3b97-8bc1-8da6f0b4a9a8,Eustolia154 Barton704,2011-01-15,female,9,Atopic dermatitis; Otitis media; Perennial all...,6,cetirizine hydrochloride 5 MG Oral Tablet; NDA...
32,03777c32-ed98-50e2-f75d-cbcad532c610,Eloise59 Gerhold939,1968-03-26,female,7,Body mass index 30+ - obesity (finding); Local...,10,Naproxen sodium 220 MG Oral Tablet; Acetaminop...
5,0a168e32-7b62-8597-0e11-296871bb764f,Bryon392 Howell947,1998-12-09,male,7,Appendicitis; History of appendectomy; Acute b...,5,Acetaminophen 325 MG Oral Tablet; Ibuprofen 20...
6,122b4063-18dc-f20f-204c-6f2da758717b,Carlota980 Morales679,2018-09-16,female,1,Viral sinusitis (disorder),1,Amoxicillin 250 MG / Clavulanate 125 MG Oral T...
30,1293efbb-72dc-8b15-3bd6-916f96ed54f4,Chauncey770 Jones311,1959-10-18,male,12,Sepsis (disorder); Body mass index 30+ - obesi...,3,24 HR Metformin hydrochloride 500 MG Extended ...
14,162135b4-931d-a531-c2ad-047b0c24abe8,Genna99 Stehr398,2006-03-14,female,6,Perennial allergic rhinitis with seasonal vari...,10,cetirizine hydrochloride 5 MG Oral Tablet; NDA...
8,1ea40a05-c295-8b66-66a2-fda3bc1ed13f,Brock407 Paucek755,1960-05-25,male,7,Received certificate of high school equivalenc...,4,ferrous sulfate 325 MG Oral Tablet; Acetaminop...
25,1f06f787-61cd-dd2d-a6b2-347507100f10,Noel608 Quigley282,1974-07-07,male,8,Rupture of appendix; Appendicitis; History of ...,2,Acetaminophen 325 MG Oral Tablet; Amoxicillin ...
39,246fb368-8991-dc93-f6a5-eca807e7dbde,Eleni953 Hilpert278,1990-10-28,female,6,Hypertension; Chronic neck pain (finding); Nor...,5,diphenhydrAMINE Hydrochloride 25 MG Oral Table...
26,355a3cad-a055-c14b-117e-47accfc708ca,Carole948 Konopelski743,1991-01-12,female,13,Cerebral palsy (disorder); Pain (finding); Poo...,3,Errin 28 Day Pack; Jolivette 28 Day Pack; 1 ML...


<IPython.core.display.JSON object>

### 4C) Charger le modèle partagé pour finaliser le `gold_source` et alimenter les étapes suivantes

Le même modèle sert ensuite :
- au filtrage LLM final des conditions
- à la génération des notes
- à la reconstruction
- au scoring sémantique

### 4C.1) Charger MedGemma une seule fois

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

@dataclass
class LLMResult:
    note: str
    used_llm: bool
    model_name: str

class MedGemma:
    def __init__(self, model_name: str):
        self.model_name = model_name
        self._pipe = None

    def available(self) -> bool:
        try:
            import transformers  # noqa: F401
            import torch  # noqa: F401
            return True
        except Exception:
            return False

    def load(self) -> None:
        token = os.environ.get("HF_TOKEN") or None
        tokenizer = AutoTokenizer.from_pretrained(self.model_name, token=token)

        model = None
        try:
            model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                token=token,
                torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
                device_map="auto",
            )
        except Exception:
            # 4-bit fallback
            bnb = BitsAndBytesConfig(load_in_4bit=True)
            model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                token=token,
                quantization_config=bnb,
                device_map="auto",
            )

        try:
            model.generation_config.max_length = None
        except Exception:
            pass

        self._pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")

    def generate(self, prompt: str, max_new_tokens: int, temperature: float, do_sample: bool) -> str:
        if self._pipe is None:
            self.load()
        out = self._pipe(
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=do_sample,
            return_full_text=False,
        )
        return out[0]["generated_text"]

medgemma = MedGemma(MODEL_NAME)

### 4D) Finaliser le `gold_source` avec un filtre LLM sur les conditions

Cette étape garde uniquement les conditions médicales pertinentes à partir de la source préfiltrée.

In [12]:
# =========================
# Step B: LLM patient-level filtering (selected sample only)
# =========================

def _strip_md_fences(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

def _extract_first_json(text: str) -> Optional[str]:
    text = _strip_md_fences(text)
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    return None

def _try_load_json(s: str) -> Optional[dict]:
    try:
        return json.loads(s)
    except Exception:
        return None

def _repair_json(s: str) -> str:
    s = re.sub(r",\s*}", "}", s)
    s = re.sub(r",\s*]", "]", s)
    return s

def llm_keep_condition_indexes(patient_id: str, candidates: List[dict]) -> List[int]:
    """Return indexes to keep (medical conditions only). Cache on disk. Never raises."""
    # Cache key based on candidate content
    key_payload = {
        "patient_id": patient_id,
        "candidates": [{"system": c.get("system"), "code": c.get("code"), "display": c.get("display")} for c in candidates],
    }
    key_str = json.dumps(key_payload, ensure_ascii=False, sort_keys=True)
    import hashlib
    cache_id = hashlib.sha1(key_str.encode('utf-8')).hexdigest()[:16]
    cache_path = FILTER_CACHE_DIR / f"llm_keepidx_{patient_id}_{cache_id}.json"
    if cache_path.exists():
        obj = safe_read_json(cache_path)
        if obj and isinstance(obj.get("keep_indexes"), list):
            return [int(x) for x in obj["keep_indexes"] if str(x).isdigit()]

    # Build numbered list
    lines = []
    for i, c in enumerate(candidates):
        sys = c.get("system") or ""
        code = c.get("code") or ""
        disp = c.get("display") or ""
        lines.append(f"{i}. [{sys}|{code}] {disp}")
    items_txt = "\n".join(lines) if lines else ""

    schema = {
        "keep_indexes": ["int"],
        "drop_indexes": ["int"],
        "reasons": {"<index>": "medical|sdoh|admin|legal|behavior|other"},
    }

    prompt = f"""You are validating patient FHIR Conditions from synthetic data.

Goal:
Keep ONLY true medical conditions (diagnoses, diseases, symptoms, disorders, injuries).
Drop social determinants (employment, housing, education, violence exposure), legal/admin items, and similar non-medical content.

Rules:
- DO NOT invent items.
- Decide only by selecting indexes from the list.
- If unsure whether something is medical, KEEP it.

Return ONLY valid JSON (no markdown, no commentary).
The first character must be '{{' and the last must be '}}'.
Use exactly this schema:
{json.dumps(schema, indent=2)}

Condition candidates:
{items_txt}

JSON:"""

    # If LLM not available, keep all
    if not (USE_LLM_PATIENT_FILTER and medgemma.available()):
        keep = list(range(len(candidates)))
        cache_path.write_text(json.dumps({"keep_indexes": keep, "mode": "no_llm"}, ensure_ascii=False, indent=2), encoding="utf-8")
        return keep

    raw = ""
    try:
        raw = medgemma.generate(
            prompt,
            max_new_tokens=LLM_FILTER_MAX_NEW_TOKENS,
            temperature=LLM_FILTER_TEMPERATURE,
            do_sample=LLM_FILTER_DO_SAMPLE,
        )
    except Exception:
        keep = list(range(len(candidates)))
        cache_path.write_text(json.dumps({"keep_indexes": keep, "mode": "llm_error"}, ensure_ascii=False, indent=2), encoding="utf-8")
        return keep

    # Parse
    js = _extract_first_json(raw) or ""
    obj = _try_load_json(js) or _try_load_json(_repair_json(js)) if js else None

    # Retry once with a fix prompt if needed
    if obj is None and LLM_FILTER_RETRIES > 0:
        fix_prompt = f"""Return ONLY valid JSON matching this schema:
{json.dumps(schema, indent=2)}

Convert the following text to valid JSON. Do not add commentary.
Text:
{raw}

JSON:"""
        try:
            fixed = medgemma.generate(
                fix_prompt,
                max_new_tokens=LLM_FILTER_MAX_NEW_TOKENS,
                temperature=0.0,
                do_sample=False,
            )
            js2 = _extract_first_json(fixed) or ""
            obj = _try_load_json(js2) or _try_load_json(_repair_json(js2)) if js2 else None
        except Exception:
            obj = None

    keep_indexes: List[int] = []
    if isinstance(obj, dict) and isinstance(obj.get("keep_indexes"), list):
        for x in obj["keep_indexes"]:
            try:
                ix = int(x)
                if 0 <= ix < len(candidates):
                    keep_indexes.append(ix)
            except Exception:
                continue

    # Safety: if empty or invalid, keep all
    if not keep_indexes:
        keep_indexes = list(range(len(candidates)))

    # Ensure deterministic order, unique
    keep_indexes = sorted(set(keep_indexes))

    cache_path.write_text(
        json.dumps({"keep_indexes": keep_indexes, "raw": raw[:2000]}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return keep_indexes

def apply_llm_filter_to_sample(patients_in: List[dict]) -> List[dict]:
    out = []
    for p in patients_in:
        pid = p["patient_id"]
        cands = p.get("conditions_candidates") or []
        keep_idx = llm_keep_condition_indexes(pid, cands)
        kept = [cands[i] for i in keep_idx if i < len(cands)]
        kept_disp = [x.get("display") for x in kept if x.get("display")]

        # Final patient record for downstream steps
        p2 = dict(p)
        p2["conditions_llm"] = kept_disp
        p2["conditions"] = list(dict.fromkeys(kept_disp))[:MAX_CONDITIONS_PER_PATIENT]
        # keep candidates for audit
        out.append(p2)
    return out

# Apply LLM filter to the selected sample and write gold
patients = apply_llm_filter_to_sample(patients_prefilter)

df_gold = pd.DataFrame([{
    "patient_id": p["patient_id"],
    "full_name": p.get("full_name"),
    "birthDate": p.get("birthDate"),
    "gender": p.get("gender"),
    "n_conditions_candidates": len(p.get("conditions_candidates") or []),
    "n_conditions_kept": len(p.get("conditions_llm") or []),
    "conditions_preview": "; ".join((p.get("conditions") or [])[:PREVIEW_N]),
    "n_medication_requests": len(p.get("medication_requests") or []),
    "medreq_preview": "; ".join((p.get("medication_requests") or [])[:PREVIEW_N]),
} for p in patients]).sort_values("patient_id")

# Save gold (after LLM filter)
for p in patients:
    (GOLD_DIR / f'{p["patient_id"]}.json').write_text(json.dumps(p, ensure_ascii=False, indent=2), encoding="utf-8")

print("Gold dataset (after hybrid filter: prefilter then LLM)")
display(df_gold)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` 

Gold dataset (after hybrid filter: prefilter then LLM)


,patient_id,full_name,birthDate,gender,n_conditions_candidates,n_conditions_kept,conditions_preview,n_medication_requests,medreq_preview
29,01f8bbfd-cfc6-3b97-8bc1-8da6f0b4a9a8,Eustolia154 Barton704,2011-01-15,female,9,9,Atopic dermatitis; Otitis media; Perennial all...,6,cetirizine hydrochloride 5 MG Oral Tablet; NDA...
32,03777c32-ed98-50e2-f75d-cbcad532c610,Eloise59 Gerhold939,1968-03-26,female,7,5,Body mass index 30+ - obesity (finding); Local...,10,Naproxen sodium 220 MG Oral Tablet; Acetaminop...
5,0a168e32-7b62-8597-0e11-296871bb764f,Bryon392 Howell947,1998-12-09,male,7,7,Appendicitis; History of appendectomy; Acute b...,5,Acetaminophen 325 MG Oral Tablet; Ibuprofen 20...
6,122b4063-18dc-f20f-204c-6f2da758717b,Carlota980 Morales679,2018-09-16,female,1,1,Viral sinusitis (disorder),1,Amoxicillin 250 MG / Clavulanate 125 MG Oral T...
30,1293efbb-72dc-8b15-3bd6-916f96ed54f4,Chauncey770 Jones311,1959-10-18,male,12,12,Sepsis (disorder); Body mass index 30+ - obesi...,3,24 HR Metformin hydrochloride 500 MG Extended ...
14,162135b4-931d-a531-c2ad-047b0c24abe8,Genna99 Stehr398,2006-03-14,female,6,5,Childhood asthma; Otitis media; Fracture sublu...,10,cetirizine hydrochloride 5 MG Oral Tablet; NDA...
8,1ea40a05-c295-8b66-66a2-fda3bc1ed13f,Brock407 Paucek755,1960-05-25,male,7,4,Anemia (disorder); Acute bronchitis (disorder)...,4,ferrous sulfate 325 MG Oral Tablet; Acetaminop...
25,1f06f787-61cd-dd2d-a6b2-347507100f10,Noel608 Quigley282,1974-07-07,male,8,7,Rupture of appendix; Appendicitis; History of ...,2,Acetaminophen 325 MG Oral Tablet; Amoxicillin ...
39,246fb368-8991-dc93-f6a5-eca807e7dbde,Eleni953 Hilpert278,1990-10-28,female,6,6,Hypertension; Chronic neck pain (finding); Nor...,5,diphenhydrAMINE Hydrochloride 25 MG Oral Table...
26,355a3cad-a055-c14b-117e-47accfc708ca,Carole948 Konopelski743,1991-01-12,female,13,13,Cerebral palsy (disorder); Pain (finding); Poo...,3,Errin 28 Day Pack; Jolivette 28 Day Pack; 1 ML...


## 5) Génération des notes non structurées à partir du `gold_source`

Cette étape produit :
- `raw_note_text` : sortie brute du modèle
- `note_text` : note nettoyée utilisée ensuite pour la reconstruction

Le display principal de cette section est maintenant une **vue de génération fusionnée** qui combine `gold_source` et `df_notes` dans un seul tableau plus lisible.

Protocole final retenu avant benchmark : chaque note doit aussi **mentionner explicitement le nom du patient, la date de naissance et le genre**, afin que les métriques démographiques soient cohérentes avec le contenu réellement demandé à la note.

In [13]:
import ast

NOTE_STYLE_CATALOG = {
    "consult_note_short": {
        "label": "Short consultation note",
        "style_instructions": [
            "Write a short consultation note.",
            "Use natural clinical prose.",
            "Keep the text compact."
        ],
    },
    "medical_history_note": {
        "label": "Medical history note",
        "style_instructions": [
            "Write the note as a brief medical history summary.",
            "Focus on known conditions first, then medications.",
            "Use a slightly retrospective tone."
        ],
    },
    "telegraphic_note": {
        "label": "Telegraphic note",
        "style_instructions": [
            "Write like a quick doctor note.",
            "Use short fragments and abbreviated style.",
            "The text can feel compressed and not fully polished."
        ],
    },
    "health_check_summary": {
        "label": "Health check summary",
        "style_instructions": [
            "Write the note as a brief health check conclusion.",
            "Use a summarizing tone.",
            "Keep it concise and readable."
        ],
    },
}

NOTE_STYLE_NAMES = list(NOTE_STYLE_CATALOG.keys())

def assign_note_styles(patients_in: List[dict], note_styles: List[str]) -> List[dict]:
    out = []
    for i, patient in enumerate(patients_in):
        style_name = note_styles[i % len(note_styles)]
        style_cfg = NOTE_STYLE_CATALOG[style_name]
        p2 = dict(patient)
        p2["note_style"] = style_name
        p2["note_style_label"] = style_cfg["label"]
        out.append(p2)
    return out

def note_prompt(patient: dict) -> str:
    note_style = patient.get("note_style", NOTE_STYLE_NAMES[0])
    style_cfg = NOTE_STYLE_CATALOG[note_style]
    style_block = "\n".join([f"- {x}" for x in style_cfg["style_instructions"]])

    name = patient.get("full_name") or ""
    dob = patient.get("birthDate") or ""
    gender = patient.get("gender") or ""

    cond_lines = "\n".join([f"- {c}" for c in patient.get("conditions", [])]) or "- None"
    med_lines = "\n".join([f"- {m}" for m in patient.get("medication_requests", [])]) or "- None"

    return f"""You are a doctor writing a free-text clinical note.

Style requirements:
{style_block}

Hard constraints:
- The note must explicitly mention the patient name in a natural way.
- The note must explicitly mention the date of birth in a natural way.
- The note must explicitly mention the gender in a natural way.
- Use all listed conditions and all listed medication requests somewhere in the note.
- Do not omit any listed item.
- Do not add any new diagnosis, symptom, medication, treatment, or contextual detail.
- Keep the note non-structured and natural.

Patient:
- Name: {name}
- Date of birth: {dob}
- Gender: {gender}

Conditions:
{cond_lines}

Medication requests:
{med_lines}

Clinical note:"""

def template_note(patient: dict) -> str:
    note_style = patient.get("note_style", NOTE_STYLE_NAMES[0])
    name = patient.get("full_name") or ""
    dob = patient.get("birthDate") or ""
    gender = patient.get("gender") or ""

    cond_txt = "; ".join(patient.get("conditions", [])) or "None"
    med_txt = "; ".join(patient.get("medication_requests", [])) or "None"
    prefix = {
        "consult_note_short": "Consultation note",
        "medical_history_note": "Medical history",
        "telegraphic_note": "Quick note",
        "health_check_summary": "Health check summary",
    }.get(note_style, "Clinical note")
    return (
        f"{prefix}: {name} (DOB: {dob}, gender: {gender}) with {cond_txt}. "
        f"Medication requests include {med_txt}."
    )

def shorten(s: str, n: int = 220) -> str:
    s = (s or "").replace("\n", " ").strip()
    return s[:n] + ("..." if len(s) > n else "")

def _literal_string_or_none(s: str) -> Optional[str]:
    s = (s or "").strip()
    if not s:
        return None
    try:
        val = ast.literal_eval(s)
        if isinstance(val, str):
            return val
    except Exception:
        pass
    return None

def _dedupe_sentences(text: str) -> str:
    text = (text or "").strip()
    if not text:
        return ""
    parts = re.split(r"(?<=[.!?])\s+", text)
    out, seen = [], set()
    for part in parts:
        p = re.sub(r"\s+", " ", (part or "").strip())
        if not p:
            continue
        k = norm_text(p)
        if k and k not in seen:
            out.append(p)
            seen.add(k)
    return " ".join(out).strip() if out else re.sub(r"\s+", " ", text).strip()

def _normalize_candidate(text: str) -> str:
    s = (text or "").replace("\r\n", "\n").replace("\r", "\n").strip()
    if not s:
        return ""

    s = re.sub(r"\\boxed\{([^}]*)\}", r"\1", s, flags=re.S)
    s = s.replace("\\n", " ").replace("\\t", " ")
    s = re.sub(r"^Clinical note:\s*", "", s, flags=re.I)
    s = re.sub(r"^Final Answer:\s*", "", s, flags=re.I)
    s = re.sub(r"^The final answer is\s*", "", s, flags=re.I)
    s = s.strip().strip('"').strip("'").strip()

    cut_patterns = [
        r"\n\s*Final Answer\s*:",
        r"\n\s*```",
        r"\n\s*print\s*\(",
        r"\n\s*Patient\s*:",
        r"\n\s*Conditions\s*:",
        r"\n\s*Medication requests\s*:",
        r"\n\s*Diagnoses\s*\(Condition\)\s*:",
        r"\n\s*Medications\s*\(MedicationRequest\)\s*:",
    ]
    cut_idx = len(s)
    for pat in cut_patterns:
        m = re.search(pat, s, flags=re.I)
        if m:
            cut_idx = min(cut_idx, m.start())
    s = s[:cut_idx].strip()

    s = re.sub(r"\s+", " ", s).strip()
    s = _dedupe_sentences(s)

    lower = s.lower()
    if any(tok in lower for tok in ["```", "print(", "final answer", "\\boxed{"]):
        return ""
    return s

def _candidate_score(text: str) -> float:
    s = (text or "").strip()
    if not s:
        return -1e9
    score = float(len(s))
    lower = s.lower()
    bad_tokens = [
        "final answer", "```", "print(", "\\boxed{",
        "patient:", "conditions:", "medication requests:",
        "diagnoses (condition):", "medications (medicationrequest):",
    ]
    score -= 200.0 * sum(tok in lower for tok in bad_tokens)
    score += 25.0 * bool(re.search(r"[A-Za-z]", s))
    score += 25.0 * (len(s.split()) >= 8)
    score += 25.0 * bool(re.search(r"[.!?]", s))
    return score

def clean_generated_note(raw_text: str) -> str:
    raw = (raw_text or "").strip()
    if not raw:
        return ""

    raw = raw.replace("\r\n", "\n").replace("\r", "\n").strip()

    candidates = [raw]

    if re.search(r"Clinical note\s*:", raw, flags=re.I):
        candidates.append(re.split(r"Clinical note\s*:", raw, flags=re.I)[-1])

    for pat in [r"\n\s*Final Answer\s*:", r"\n\s*```", r"\n\s*print\s*\("]:
        m = re.search(pat, raw, flags=re.I)
        if m:
            candidates.append(raw[:m.start()])

    for m in re.finditer(r"\\boxed\{([^}]*)\}", raw, flags=re.S):
        candidates.append(m.group(1))

    for pat in [r'print\((?P<q>"(?:[^"\\]|\\.)*")\)', r"print\((?P<q>'(?:[^'\\]|\\.)*')\)"]:
        for m in re.finditer(pat, raw, flags=re.S):
            lit = _literal_string_or_none(m.group("q"))
            if lit:
                candidates.append(lit)

    for pat in [r'"""{3}(.*?)"""{3}'.replace('"""{3}', '"""'), r"'{3}(.*?)'{3}"]:
        for m in re.finditer(pat, raw, flags=re.S):
            block = (m.group(1) or "").strip()
            if block:
                candidates.append(block)

    cleaned = []
    seen = set()
    for cand in candidates:
        c = _normalize_candidate(cand)
        k = norm_text(c)
        if c and k and k not in seen:
            cleaned.append(c)
            seen.add(k)

    if not cleaned:
        return re.sub(r"\s+", " ", raw).strip()

    best = max(cleaned, key=_candidate_score)
    return best

# Assign a controlled note style to each patient before generation.
patients = assign_note_styles(patients, NOTE_STYLE_NAMES)

# Update gold files and the gold dataframe so note_style becomes part of the canonical source.
for p in patients:
    (GOLD_DIR / f'{p["patient_id"]}.json').write_text(json.dumps(p, ensure_ascii=False, indent=2), encoding="utf-8")

df_gold = pd.DataFrame([{
    "patient_id": p["patient_id"],
    "full_name": p.get("full_name"),
    "birthDate": p.get("birthDate"),
    "gender": p.get("gender"),
    "note_style": p.get("note_style"),
    "note_style_label": p.get("note_style_label"),
    "n_conditions_candidates": len(p.get("conditions_candidates") or []),
    "n_conditions_kept": len(p.get("conditions_llm") or []),
    "conditions_preview": "; ".join((p.get("conditions") or [])[:PREVIEW_N]),
    "n_medication_requests": len(p.get("medication_requests") or []),
    "medreq_preview": "; ".join((p.get("medication_requests") or [])[:PREVIEW_N]),
} for p in patients]).sort_values(["note_style", "patient_id"])

print("Gold dataset updated with note_style")

print("Note style distribution")
display(df_gold["note_style"].value_counts().to_frame("n"))

RAW_NOTES_DIR.mkdir(parents=True, exist_ok=True)

raw_notes: Dict[str, str] = {}
notes: Dict[str, LLMResult] = {}
for p in patients:
    pid = p["patient_id"]
    if medgemma.available():
        try:
            txt = medgemma.generate(
                note_prompt(p),
                max_new_tokens=NOTE_GENERATION_MAX_NEW_TOKENS,
                temperature=NOTE_GENERATION_TEMPERATURE,
                do_sample=NOTE_GENERATION_DO_SAMPLE
            )
            raw_txt = (txt or "").strip()
            clean_txt = clean_generated_note(raw_txt) or raw_txt
            raw_notes[pid] = raw_txt
            notes[pid] = LLMResult(note=clean_txt, used_llm=True, model_name=MODEL_NAME)
        except Exception:
            fallback = template_note(p)
            raw_notes[pid] = fallback
            notes[pid] = LLMResult(note=fallback, used_llm=False, model_name="template")
    else:
        fallback = template_note(p)
        raw_notes[pid] = fallback
        notes[pid] = LLMResult(note=fallback, used_llm=False, model_name="template")

for pid, res in notes.items():
    (RAW_NOTES_DIR / f"{pid}.txt").write_text(raw_notes.get(pid, res.note), encoding="utf-8")
    (NOTES_DIR / f"{pid}.txt").write_text(res.note, encoding="utf-8")

gold_by_pid = {p["patient_id"]: p for p in patients}
df_notes = pd.DataFrame([{
    "patient_id": pid,
    "full_name": (gold_by_pid.get(pid, {}) or {}).get("full_name"),
    "birthDate": (gold_by_pid.get(pid, {}) or {}).get("birthDate"),
    "gender": (gold_by_pid.get(pid, {}) or {}).get("gender"),
    "note_style": (gold_by_pid.get(pid, {}) or {}).get("note_style"),
    "note_style_label": (gold_by_pid.get(pid, {}) or {}).get("note_style_label"),
    "used_llm": res.used_llm,
    "model": res.model_name,
    "raw_note_preview": shorten(raw_notes.get(pid, "")),
    "note_text": res.note,
    "note_preview": shorten(res.note),
    "raw_chars": len(raw_notes.get(pid, "")),
    "clean_chars": len(res.note or ""),
} for pid, res in notes.items()]).sort_values(["note_style", "patient_id"])

# If we do not have gold_by_pid yet, fallback to patients list for demographics
if df_notes["full_name"].isna().all():
    demo = pd.DataFrame([{
        "patient_id": p["patient_id"],
        "full_name": p.get("full_name"),
        "birthDate": p.get("birthDate"),
        "gender": p.get("gender"),
        "note_style": p.get("note_style"),
        "note_style_label": p.get("note_style_label"),
    } for p in patients])
    df_notes = df_notes.drop(columns=["full_name", "birthDate", "gender", "note_style", "note_style_label"]).merge(demo, on="patient_id", how="left")
    df_notes = df_notes[[
        "patient_id", "full_name", "birthDate", "gender",
        "note_style", "note_style_label",
        "used_llm", "model", "raw_note_preview", "note_text", "note_preview",
        "raw_chars", "clean_chars"
    ]].sort_values(["note_style", "patient_id"])

print("Notes dataset (note_text = cleaned note used for reconstruction)")
display(df_notes)

Gold dataset updated with note_style
Note style distribution


,n
note_style,
consult_note_short,10
health_check_summary,10
medical_history_note,10
telegraphic_note,10


Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Notes dataset (note_text = cleaned note used for reconstruction)


,patient_id,full_name,birthDate,gender,note_style,note_style_label,used_llm,model,raw_note_preview,note_text,note_preview,raw_chars,clean_chars
32,03777c32-ed98-50e2-f75d-cbcad532c610,Eloise59 Gerhold939,1968-03-26,female,consult_note_short,Short consultation note,True,google/medgemma-4b-it,Eloise59 Gerhold939 presented today complainin...,Eloise59 Gerhold939 presented today complainin...,Eloise59 Gerhold939 presented today complainin...,1188,991
8,1ea40a05-c295-8b66-66a2-fda3bc1ed13f,Brock407 Paucek755,1960-05-25,male,consult_note_short,Short consultation note,True,google/medgemma-4b-it,Brock407 Paucek755 presented today with a comp...,Brock407 Paucek755 presented today with a comp...,Brock407 Paucek755 presented today with a comp...,1162,612
0,51c7ff6a-33e3-3e1b-d3ad-0035c8227dfa,Georgiann138 Heaney114,2002-04-11,female,consult_note_short,Short consultation note,True,google/medgemma-4b-it,Georgiann138 Heaney114 presented today. She is...,Georgiann138 Heaney114 presented today. She is...,Georgiann138 Heaney114 presented today. She is...,1029,942
12,752a3c69-44d7-d6af-0340-7d3e748f6060,Oralia106 Williamson769,2000-08-10,female,consult_note_short,Short consultation note,True,google/medgemma-4b-it,"Oralia Williamson, a 22-year-old female patien...","Oralia Williamson, a 22-year-old female patien...","Oralia Williamson, a 22-year-old female patien...",1309,420
20,7d28d76a-9ac8-67b4-3c88-0a75be3d0851,Erna640 Robel940,1969-01-03,female,consult_note_short,Short consultation note,True,google/medgemma-4b-it,Erna640 Robel940 presented today with complain...,Erna640 Robel940 presented today with complain...,Erna640 Robel940 presented today with complain...,964,964
28,84fee56a-9ef7-8cff-e7d3-526ece562d87,Dewitt635 Bernier607,1953-09-21,male,consult_note_short,Short consultation note,True,google/medgemma-4b-it,Dewitt635 Bernier607 presented today with acut...,Dewitt635 Bernier607 presented today with acut...,Dewitt635 Bernier607 presented today with acut...,1195,610
24,8c6ae452-5f8c-9ff6-006d-c6c860acf5cd,Darla154 Hoppe518,1917-05-15,female,consult_note_short,Short consultation note,True,google/medgemma-4b-it,"Darla154 Hoppe518, female born 1917-05-15, pre...","Darla154 Hoppe518, female born 1917-05-15, pre...","Darla154 Hoppe518, female born 1917-05-15, pre...",1041,1039
36,ce1153cb-d4ad-77e2-cd07-575e249a83ad,Deidra675 Rosenbaum794,2006-10-18,female,consult_note_short,Short consultation note,True,google/medgemma-4b-it,Deidra675 Rosenbaum794 presented today with co...,Deidra675 Rosenbaum794 presented today with co...,Deidra675 Rosenbaum794 presented today with co...,1133,831
16,dc6c06d0-a7d8-100f-c08b-46c93700c188,Taylor21 Gulgowski816,2006-07-11,male,consult_note_short,Short consultation note,True,google/medgemma-4b-it,Taylor21 Gulgowski816 presented today with com...,Taylor21 Gulgowski816 presented today with com...,Taylor21 Gulgowski816 presented today with com...,1202,353
4,fabb7ac2-1c62-3293-f43c-e35fb903c1c7,Nicholas495 Muller251,1988-08-29,male,consult_note_short,Short consultation note,True,google/medgemma-4b-it,Nicholas495 Muller251 presented today with acu...,Nicholas495 Muller251 presented today with acu...,Nicholas495 Muller251 presented today with acu...,1131,672


## 6) Reconstruction structurée depuis `note_text`

Version de lecture :
- `recon_raw` = extraction brute du LLM
- `recon_clean` = reconstruction principale utilisée pour le scoring
- `recon_clipped` = vue debug stricte par overlap exact

In [14]:

class Extracted(BaseModel):
    # Raw extraction from LLM (for scoring/debug and inspection only).
    # We do NOT trust demographics for the final reconstruction to avoid cross-patient swaps.
    patient_name: Optional[str] = None
    date_of_birth: Optional[str] = None
    gender: Optional[str] = None
    conditions: List[str] = Field(default_factory=list)
    medication_requests: List[str] = Field(default_factory=list)

SCHEMA = {
    "patient_name": "string or null",
    "date_of_birth": "YYYY-MM-DD or null",
    "gender": "male|female|other|unknown or null",
    "conditions": ["string"],
    "medication_requests": ["string"],
}

def extraction_prompt(note_text: str) -> str:
    example_output = {
        "patient_name": "John Smith",
        "date_of_birth": "1980-04-12",
        "gender": "male",
        "conditions": [
            "Hypertension",
            "Type 2 diabetes mellitus"
        ],
        "medication_requests": [
            "Lisinopril 10 MG Oral Tablet",
            "Metformin 500 MG Oral Tablet"
        ]
    }

    return f"""You are a clinical information extraction engine.

Your task is to read ONE clinical note and return EXACTLY ONE valid JSON object.

Return rules:
- Return JSON only.
- No markdown.
- No code fences.
- No explanation.
- No comments.
- No extra keys.
- The first character must be '{{' and the last character must be '}}'.

You must return exactly these keys:
- patient_name
- date_of_birth
- gender
- conditions
- medication_requests

Extraction rules:
- Extract only information explicitly written in the note.
- Do not infer, guess, interpret, or add medical knowledge.
- Do not invent diagnoses or medications.
- For conditions and medication_requests, copy the wording from the note as closely as possible.
- Do not rewrite items into synonyms unless the note itself uses that wording.
- Each list element must contain exactly one item, not a sentence.
- Split combined items into separate list elements when clearly separable.
- Remove duplicates.
- Preserve medication strength, form, and dosage only if explicitly present in the note.

Field rules:
- patient_name: the patient full name if explicitly present, otherwise null.
- date_of_birth: only YYYY-MM-DD if explicitly present, otherwise null.
- gender: must be exactly one of ["male", "female", "other", "unknown"] or null.
- conditions: include only diagnoses, disorders, chronic diseases, or medical problems explicitly stated.
- medication_requests: include only medications explicitly prescribed, active, continued, or listed as current medications.

Do NOT put these in conditions unless the note explicitly states they are diagnoses:
- symptoms alone
- procedures
- laboratory tests
- imaging findings alone
- allergies
- vaccinations
- family history
- social history
- employment
- education
- stress
- risk activity
- insurance or administrative details

Do NOT put these in medication_requests:
- procedures
- tests
- follow-up instructions
- lifestyle advice
- allergies
- vaccines

Missing value rules:
- If a scalar field is missing, use null.
- If a list field is missing, use [].

Schema:
{json.dumps(SCHEMA, ensure_ascii=False, indent=2)}

Valid output example:
{json.dumps(example_output, ensure_ascii=False, indent=2)}

Clinical note:
<note>
{note_text}
</note>

JSON:"""

def strip_fences(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

def extract_first_json(text: str) -> Optional[str]:
    text = strip_fences(text)
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None

def try_load(s: str) -> Optional[dict]:
    try:
        return json.loads(s)
    except Exception:
        return None

def repair_json(s: str) -> str:
    s = re.sub(r",\s*}", "}", s)
    s = re.sub(r",\s*]", "]", s)
    if '"' not in s and "'" in s:
        s = s.replace("'", '"')
    return s

def fix_to_json(raw_text: str) -> str:
    fix_prompt = f"""Convert the text below into EXACTLY ONE valid JSON object.

Rules:
- Return JSON only.
- No markdown.
- No explanation.
- No extra keys.
- Do not invent information.
- Keep only information already present in the text.
- If a scalar field is missing, use null.
- If a list field is missing, use [].
- gender must be one of ["male", "female", "other", "unknown"] or null.
- date_of_birth must be YYYY-MM-DD or null.

Schema:
{json.dumps(SCHEMA, ensure_ascii=False, indent=2)}

Text:
<text>
{raw_text}
</text>

JSON:"""
    return medgemma.generate(fix_prompt, max_new_tokens=RECON_FIX_JSON_MAX_NEW_TOKENS, temperature=RECON_TEMPERATURE, do_sample=RECON_DO_SAMPLE)

def parse_llm_json(llm_text: str, retries: int = 1) -> Optional[dict]:
    js = extract_first_json(llm_text)
    if js:
        obj = try_load(js)
        if obj is not None:
            return obj
        obj = try_load(repair_json(js))
        if obj is not None:
            return obj
    if retries > 0 and medgemma.available():
        fixed = fix_to_json(llm_text)
        return parse_llm_json(fixed, retries=retries - 1)
    return None

def ensure_list(x) -> List[str]:
    if isinstance(x, list):
        return x
    return []

def clean_extracted_item(v: str) -> str:
    s = str(v or "").strip()
    s = re.sub(r"^\s*[-*•]+\s*", "", s)
    s = re.sub(r"^\s*\d+[\.\)\-:]\s*", "", s)
    s = re.sub(r"\s+", " ", s)
    s = s.strip(" \t\r\n,;")
    return s

def is_noisy_candidate(k: str, filter_noisy: bool) -> bool:
    if not filter_noisy or not PREFILTER_ENABLED:
        return False

    if not k:
        return True

    noisy_phrases = [norm_text(x) for x in (PREFILTER_DROP_SUBSTRINGS or []) if x]
    strong_multiword = [x for x in noisy_phrases if " " in x]
    short_singletons = set([x for x in noisy_phrases if " " not in x])

    # Strong multiword phrases can be filtered by substring.
    if any(p in k for p in strong_multiword):
        return True

    # Single-word categories are only filtered on near-exact / very short administrative entries.
    if k in short_singletons:
        return True

    return False

def postprocess_list(values: List[str], max_items: int, filter_noisy: bool) -> List[str]:
    out = []
    seen = set()

    for v in ensure_list(values):
        cleaned = clean_extracted_item(v)
        k = norm_text(cleaned)

        if not k:
            continue
        if len(k) <= 1:
            continue
        if len(k) > 160:
            continue
        if is_noisy_candidate(k, filter_noisy):
            continue
        if k in seen:
            continue

        seen.add(k)
        out.append(cleaned)

        if len(out) >= max_items:
            break

    return out

def clip_to_gold(values: List[str], gold_values: List[str]) -> List[str]:
    """Strict exact-normalized overlap with the patient gold list. Debug only, not used for main scoring."""
    allowed = set([norm_text(x) for x in (gold_values or []) if x])
    out = []
    for v in values or []:
        if norm_text(v) in allowed:
            out.append(v)
    return out

# Gold lookup (from the canonical patient records / filtered source used to generate the notes)
gold_by_pid = {p["patient_id"]: p for p in patients}

# Reconstruction outputs:
# - recon_raw: raw LLM extraction
# - recon_clean: cleaned reconstruction without strict clipping, used for the main scoring
# - recon_clipped: strict exact-overlap debug view only
# - recon: backward-compatible alias to recon_clean
recon_raw: Dict[str, dict] = {}
recon_clean: Dict[str, dict] = {}
recon_clipped: Dict[str, dict] = {}
recon: Dict[str, dict] = {}

for pid, res in notes.items():
    obj = None
    if medgemma.available():
        try:
            txt = medgemma.generate(
                extraction_prompt((res.note or "").strip()),
                max_new_tokens=RECON_EXTRACTION_MAX_NEW_TOKENS,
                temperature=RECON_TEMPERATURE,
                do_sample=RECON_DO_SAMPLE,
            )
            obj = parse_llm_json(txt, retries=1)
        except Exception:
            obj = None

    if obj is None:
        obj = {"patient_name": None, "date_of_birth": None, "gender": None, "conditions": [], "medication_requests": []}

    obj.setdefault("patient_name", None)
    obj.setdefault("date_of_birth", None)
    obj.setdefault("gender", None)
    obj["conditions"] = ensure_list(obj.get("conditions"))
    obj["medication_requests"] = ensure_list(obj.get("medication_requests"))

    gold = gold_by_pid.get(pid, {}) or {}

    # Main cleaned reconstruction used for scoring
    cond_clean = postprocess_list(obj.get("conditions") or [], MAX_CONDITIONS_PER_PATIENT, filter_noisy=True)
    medr_clean = postprocess_list(obj.get("medication_requests") or [], MAX_MEDICATION_REQUESTS_PER_PATIENT, filter_noisy=False)

    # Strict exact-overlap debug reconstruction only
    cond_clipped = clip_to_gold(cond_clean, gold.get("conditions", []))
    medr_clipped = clip_to_gold(medr_clean, gold.get("medication_requests", []))

    recon_raw[pid] = obj

    recon_clean[pid] = {
        "note_used_for_extraction": (res.note or "").strip(),
        "patient_id": pid,
        "full_name": gold.get("full_name"),
        "birthDate": gold.get("birthDate"),
        "gender": gold.get("gender"),
        "conditions": cond_clean,
        "medication_requests": medr_clean,
    }

    recon_clipped[pid] = {
        "patient_id": pid,
        "full_name": gold.get("full_name"),
        "birthDate": gold.get("birthDate"),
        "gender": gold.get("gender"),
        "conditions": cond_clipped,
        "medication_requests": medr_clipped,
    }

    recon[pid] = recon_clean[pid]

    (RECON_DIR / f"{pid}.raw.json").write_text(json.dumps(recon_raw[pid], ensure_ascii=False, indent=2), encoding="utf-8")
    (RECON_DIR / f"{pid}.json").write_text(json.dumps(recon_clean[pid], ensure_ascii=False, indent=2), encoding="utf-8")
    (RECON_DIR / f"{pid}.clipped.json").write_text(json.dumps(recon_clipped[pid], ensure_ascii=False, indent=2), encoding="utf-8")

df_recon = pd.DataFrame([{
    "patient_id": pid,
    "full_name": recon_clean[pid].get("full_name"),
    "birthDate": recon_clean[pid].get("birthDate"),
    "gender": recon_clean[pid].get("gender"),
    "raw_conditions_n": len(ensure_list(recon_raw[pid].get("conditions"))),
    "conditions_n": len(recon_clean[pid]["conditions"]),
    "conditions_clipped_n": len(recon_clipped[pid]["conditions"]),
    "raw_medreq_n": len(ensure_list(recon_raw[pid].get("medication_requests"))),
    "medreq_n": len(recon_clean[pid]["medication_requests"]),
    "medreq_clipped_n": len(recon_clipped[pid]["medication_requests"]),
    "conditions_preview": "; ".join(recon_clean[pid]["conditions"][:PREVIEW_N]),
    "conditions_clipped_preview": "; ".join(recon_clipped[pid]["conditions"][:PREVIEW_N]),
    "medreq_preview": "; ".join(recon_clean[pid]["medication_requests"][:PREVIEW_N]),
    "medreq_clipped_preview": "; ".join(recon_clipped[pid]["medication_requests"][:PREVIEW_N]),
    "raw_file": str((RECON_DIR / f"{pid}.raw.json").resolve()),
    "recon_file": str((RECON_DIR / f"{pid}.json").resolve()),
    "recon_clipped_file": str((RECON_DIR / f"{pid}.clipped.json").resolve()),
} for pid in sorted(recon_clean.keys())])

print("Reconstruction dataset (main = cleaned/no strict clipping, clipped = exact-overlap debug only)")
display(df_recon)


Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Reconstruction dataset (main = cleaned/no strict clipping, clipped = exact-overlap debug only)


,patient_id,full_name,birthDate,gender,raw_conditions_n,conditions_n,conditions_clipped_n,raw_medreq_n,medreq_n,medreq_clipped_n,conditions_preview,conditions_clipped_preview,medreq_preview,medreq_clipped_preview,raw_file,recon_file,recon_clipped_file
0,01f8bbfd-cfc6-3b97-8bc1-8da6f0b4a9a8,Eustolia154 Barton704,2011-01-15,female,8,8,3,6,6,5,atopic dermatitis; otitis media; perennial all...,atopic dermatitis; otitis media; perennial all...,cetirizine hydrochloride 5 MG Oral Tablet; Hyd...,cetirizine hydrochloride 5 MG Oral Tablet; Hyd...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...
1,03777c32-ed98-50e2-f75d-cbcad532c610,Eloise59 Gerhold939,1968-03-26,female,5,5,1,7,7,6,obesity; Coronary Heart Disease; Osteoarthriti...,Coronary Heart Disease,Clopidogrel 75 MG Oral Tablet; Acetaminophen 2...,Clopidogrel 75 MG Oral Tablet; Acetaminophen 2...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...
2,0a168e32-7b62-8597-0e11-296871bb764f,Bryon392 Howell947,1998-12-09,male,5,5,3,5,5,5,appendicitis; acute bronchitis; viral sinusiti...,appendicitis; fracture subluxation of wrist; h...,Acetaminophen 325 MG Oral Tablet; Ibuprofen 20...,Acetaminophen 325 MG Oral Tablet; Ibuprofen 20...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...
3,122b4063-18dc-f20f-204c-6f2da758717b,Carlota980 Morales679,2018-09-16,female,1,1,0,1,1,1,Viral sinusitis,,Amoxicillin 250 MG / Clavulanate 125 MG Oral T...,Amoxicillin 250 MG / Clavulanate 125 MG Oral T...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...
4,1293efbb-72dc-8b15-3bd6-916f96ed54f4,Chauncey770 Jones311,1959-10-18,male,10,10,3,3,3,3,Sepsis; Chronic sinusitis; Prediabetes; Diabet...,Prediabetes; Diabetes; Hypertension,24 HR Metformin hydrochloride 500 MG Extended ...,24 HR Metformin hydrochloride 500 MG Extended ...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...
5,162135b4-931d-a531-c2ad-047b0c24abe8,Genna99 Stehr398,2006-03-14,female,5,5,4,4,4,4,acute viral pharyngitis; childhood asthma; oti...,childhood asthma; otitis media; fracture sublu...,cetirizine hydrochloride 5 MG Oral Tablet; NDA...,cetirizine hydrochloride 5 MG Oral Tablet; NDA...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...
6,1ea40a05-c295-8b66-66a2-fda3bc1ed13f,Brock407 Paucek755,1960-05-25,male,2,2,0,4,4,3,acute viral pharyngitis; anemia,,ferrous sulfate 325 MG Oral Tablet; acetaminop...,ferrous sulfate 325 MG Oral Tablet; acetaminop...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...
7,1f06f787-61cd-dd2d-a6b2-347507100f10,Noel608 Quigley282,1974-07-07,male,5,5,2,2,2,2,appendicitis; ruptured appendix; acute viral p...,appendicitis; facial laceration,Acetaminophen 325 MG Oral Tablet; Amoxicillin ...,Acetaminophen 325 MG Oral Tablet; Amoxicillin ...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...
8,246fb368-8991-dc93-f6a5-eca807e7dbde,Eleni953 Hilpert278,1990-10-28,female,7,7,1,4,4,3,Hypertension; Chronic neck pain; Seasonal alle...,Hypertension,Lisinopril 10 MG Oral Tablet; DiphenhydrAMINE ...,Lisinopril 10 MG Oral Tablet; DiphenhydrAMINE ...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...,/content/data_bupa_conditions_medreq/reconstru...
9,355a3cad-a055-c14b-117e-47accfc708ca,Carole948 K

## 7) Scoring officiel et scoring secondaire

Convention finale de nomenclature :

- **Scoring officiel** = `note_clean` vs `recon_clean`
- **Scoring secondaire** = `gold_source` vs `recon_clean`
- **Démographie** = métriques diagnostiques séparées (`name`, `dob`, `gender`)
- **Clipped** = supprimé des métriques de lecture courante

Pour chaque scoring (`official` et `secondary`), on calcule désormais, séparément pour `conditions` et `medication_requests` :

- `exact_precision`
- `exact_recall`
- `exact_f1`
- `semantic_precision_llm`
- `semantic_recall_llm`
- `semantic_f1_llm`
- `tp`
- `fp`
- `fn`

Remarque :
- les comptes `TN` ne sont pas utilisés comme métriques principales car l’univers négatif n’est pas naturellement fermé pour des listes de conditions et de médicaments.


In [15]:

# Scoring officiel et secondaire
# - officiel: note_clean vs recon_clean
# - secondaire: gold filtered source vs recon_clean
# - démographie: diagnostic séparé
# - clipped: non utilisé dans les métriques finales

import hashlib

def metrics_from_counts(tp: int, fp: int, fn: int) -> dict:
    precision = tp / (tp + fp) if (tp + fp) else (1.0 if fn == 0 else 0.0)
    recall = tp / (tp + fn) if (tp + fn) else 1.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    if tp == 0 and fp == 0 and fn == 0:
        f1 = 1.0
    return {
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

def metrics_from_sets(gold_set: set, pred_set: set) -> dict:
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    return metrics_from_counts(tp, fp, fn)

def name_exact(gold_name: str, pred_name: str) -> int:
    return int(bool(norm_text(gold_name)) and norm_text(gold_name) == norm_text(pred_name or ""))

def dob_exact(gold_dob: str, pred_dob: str) -> int:
    g = (gold_dob or "").strip()
    p = (pred_dob or "").strip()
    return int(bool(g) and g == p)

def gender_exact(gold_gender: str, pred_gender: str) -> int:
    return int(bool(norm_text(gold_gender or "")) and norm_text(gold_gender or "") == norm_text(pred_gender or ""))

def _parse_json_obj(text: str) -> dict:
    obj = parse_llm_json(text, retries=0)
    return obj if isinstance(obj, dict) else {}

def exact_items_supported_by_note(note_text: str, source_items: List[str]) -> List[str]:
    note_norm = norm_text(note_text or "")
    supported = []
    for item in source_items or []:
        item_norm = norm_text(item)
        if item_norm and item_norm in note_norm:
            supported.append(item)
    return supported

def _llm_judge_match_index(target: str, candidates: List[str]) -> Tuple[int, float]:
    if not candidates:
        return -1, 0.0

    candidates = candidates[:25]

    prompt = (
        "You are a strict matcher for clinical concepts.\n"
        "Task: decide whether the TARGET is semantically the same as one of the CANDIDATES.\n"
        "If yes, return the index of the best candidate. If no, return -1.\n\n"
        "Rules:\n"
        "- Only match if it is clearly the same diagnosis/condition or the same medication.\n"
        "- Do not match if it is merely related or in the same family.\n"
        "- Output ONLY valid JSON. No markdown. No commentary.\n"
        "- JSON schema: {\"match_index\": integer, \"confidence\": number}\n\n"
        f"TARGET: {target}\n"
        "CANDIDATES:\n"
    )
    for i, c in enumerate(candidates):
        prompt += f"{i}. {c}\n"
    prompt += "\nJSON:"

    txt = medgemma.generate(
        prompt,
        max_new_tokens=SEMANTIC_MATCH_MAX_NEW_TOKENS,
        temperature=SCORING_TEMPERATURE,
        do_sample=SCORING_DO_SAMPLE
    )
    obj = _parse_json_obj(txt) or {}
    try:
        idx = int(obj.get("match_index", -1))
    except Exception:
        idx = -1
    try:
        conf = float(obj.get("confidence", 0.0))
    except Exception:
        conf = 0.0

    if idx < -1 or idx >= len(candidates):
        idx = -1
    if conf < 0.0:
        conf = 0.0
    if conf > 1.0:
        conf = 1.0

    return idx, conf

def _hash_lists(pred_list: List[str], gold_list: List[str]) -> str:
    payload = json.dumps({"pred": pred_list, "gold": gold_list}, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

LLM_ALIGN_DIR = SCORES_DIR / "llm_align"
LLM_ALIGN_DIR.mkdir(parents=True, exist_ok=True)

def semantic_align_and_score(pid: str, kind: str, pred_items: List[str], gold_items: List[str]) -> dict:
    """
    Secondary scoring helper: gold source vs cleaned reconstruction.
    """
    pred_items = [x for x in pred_items if x]
    gold_items = [x for x in gold_items if x]

    gold_norm = [norm_text(x) for x in gold_items]
    pred_norm = [norm_text(x) for x in pred_items]

    cache_path = LLM_ALIGN_DIR / f"{pid}.{kind}.json"
    sig = _hash_lists(pred_norm, gold_norm)

    if cache_path.exists():
        try:
            cached = json.loads(cache_path.read_text(encoding="utf-8"))
            if cached.get("signature") == sig:
                return cached
        except Exception:
            pass

    used_gold = set()
    matches = []
    tp = 0

    for p_raw, p in zip(pred_items, pred_norm):
        match_idx = -1
        conf = 1.0

        if p and p in gold_norm:
            for i, g in enumerate(gold_norm):
                if g == p and i not in used_gold:
                    match_idx = i
                    break

        if match_idx == -1 and medgemma.available():
            match_idx, conf = _llm_judge_match_index(p_raw, gold_items)
            if match_idx in used_gold:
                match_idx = -1
                conf = 0.0

        if match_idx >= 0:
            used_gold.add(match_idx)
            tp += 1

        matches.append({
            "pred": p_raw,
            "match_index": int(match_idx),
            "matched_gold": gold_items[match_idx] if match_idx >= 0 else None,
            "confidence": float(conf),
        })

    fp = len(pred_items) - tp
    fn = len(gold_items) - len(used_gold)
    metrics = metrics_from_counts(tp, fp, fn)

    out = {
        "patient_id": pid,
        "kind": kind,
        "signature": sig,
        "tp": metrics["tp"],
        "fp": metrics["fp"],
        "fn": metrics["fn"],
        "precision_semantic": metrics["precision"],
        "recall_semantic": metrics["recall"],
        "f1_semantic": metrics["f1"],
        "pred_n": len(pred_items),
        "gold_n": len(gold_items),
        "matches": matches,
    }
    cache_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
    return out

NOTE_ALIGN_DIR = SCORES_DIR / "llm_note_vs_recon"
NOTE_ALIGN_DIR.mkdir(parents=True, exist_ok=True)

def _hash_note_and_items(note_text: str, kind: str, pred_items: List[str]) -> str:
    payload = json.dumps({
        "note_text": note_text or "",
        "kind": kind,
        "pred_items": pred_items or [],
    }, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def note_vs_recon_semantic_score(pid: str, kind: str, note_text: str, pred_items: List[str]) -> dict:
    """
    Official scoring helper: note_clean vs recon_clean using LLM semantic judgment.
    """
    pred_items = [x for x in (pred_items or []) if x]
    note_text = (note_text or "").strip()

    cache_path = NOTE_ALIGN_DIR / f"{pid}.{kind}.json"
    sig = _hash_note_and_items(note_text, kind, pred_items)

    if cache_path.exists():
        try:
            cached = json.loads(cache_path.read_text(encoding="utf-8"))
            if cached.get("signature") == sig:
                return cached
        except Exception:
            pass

    if not note_text and not pred_items:
        out = {
            "patient_id": pid,
            "kind": kind,
            "signature": sig,
            "supported_n": 0,
            "unsupported_n": 0,
            "missing_n": 0,
            "pred_n": 0,
            "precision_note_consistency": 1.0,
            "recall_note_consistency": 1.0,
            "f1_note_consistency": 1.0,
            "supported_items": [],
            "unsupported_items": [],
            "missing_items": [],
        }
        cache_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
        return out

    label = "conditions" if kind == "conditions" else "medication requests"
    item_lines = "\n".join([f"{i}. {x}" for i, x in enumerate(pred_items)]) or "(none)"

    supported_indexes = []
    missing_items = []

    if medgemma.available():
        prompt = f"""You are reviewing a clinical note and a structured extracted list of {label}.

Task:
1. Decide which listed items are explicitly supported by the note or are clear semantic equivalents of what is explicitly stated.
2. Identify additional {label} explicitly present in the note but missing from the extracted list.

Rules:
- Use only what is explicitly stated in the note.
- Do not infer or invent anything.
- For conditions: count diagnoses, disorders, chronic diseases, medical problems, or explicitly stated past history items only if clearly present as such in the note.
- For medication requests: count only medications explicitly requested, prescribed, active, continued, or listed as current in the note.
- A synonym or very close canonical equivalent counts as the same item.
- Return JSON only. No markdown. No commentary.

Clinical note:
<note>
{note_text}
</note>

Extracted {label}:
{item_lines}

JSON schema:
{{"supported_indexes": [integer], "missing_items": ["string"]}}

JSON:"""
        obj = _parse_json_obj(
            medgemma.generate(
                prompt,
                max_new_tokens=NOTE_CONSISTENCY_MAX_NEW_TOKENS,
                temperature=SCORING_TEMPERATURE,
                do_sample=SCORING_DO_SAMPLE
            )
        ) or {}
        raw_idxs = obj.get("supported_indexes", [])
        if isinstance(raw_idxs, list):
            seen = set()
            for x in raw_idxs:
                try:
                    idx = int(x)
                except Exception:
                    continue
                if 0 <= idx < len(pred_items) and idx not in seen:
                    supported_indexes.append(idx)
                    seen.add(idx)
        raw_missing = obj.get("missing_items", [])
        if isinstance(raw_missing, list):
            missing_items = [str(x).strip() for x in raw_missing if str(x).strip()]

    supported_items = [pred_items[i] for i in supported_indexes]
    supported_idx_set = set(supported_indexes)
    unsupported_items = [x for i, x in enumerate(pred_items) if i not in supported_idx_set]
    missing_items = postprocess_list(
        missing_items,
        MAX_CONDITIONS_PER_PATIENT if kind == "conditions" else MAX_MEDICATION_REQUESTS_PER_PATIENT,
        filter_noisy=(kind == "conditions"),
    )

    tp = len(supported_items)
    fp = len(unsupported_items)
    fn = len(missing_items)
    metrics = metrics_from_counts(tp, fp, fn)

    out = {
        "patient_id": pid,
        "kind": kind,
        "signature": sig,
        "supported_n": metrics["tp"],
        "unsupported_n": metrics["fp"],
        "missing_n": metrics["fn"],
        "pred_n": len(pred_items),
        "precision_note_consistency": metrics["precision"],
        "recall_note_consistency": metrics["recall"],
        "f1_note_consistency": metrics["f1"],
        "supported_items": supported_items,
        "unsupported_items": unsupported_items,
        "missing_items": missing_items,
    }
    cache_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
    return out

gold_by_pid = {p["patient_id"]: p for p in patients}

score_rows = []
for pid in sorted(gold_by_pid.keys()):
    gold = gold_by_pid[pid]
    pred_clean = recon.get(pid, {}) or {}
    raw = recon_raw.get(pid, {}) or {}
    note_obj = notes.get(pid)
    note_text = (note_obj.note if note_obj else "") or ""
    note_style = gold.get("note_style")
    note_style_label = gold.get("note_style_label")

    gold_cond_list = postprocess_list(gold.get("conditions") or [], 10_000, filter_noisy=True)
    gold_med_list = postprocess_list(gold.get("medication_requests") or [], 10_000, filter_noisy=False)

    pred_cond_list = postprocess_list(pred_clean.get("conditions") or [], MAX_CONDITIONS_PER_PATIENT, filter_noisy=True)
    pred_med_list = postprocess_list(pred_clean.get("medication_requests") or [], MAX_MEDICATION_REQUESTS_PER_PATIENT, filter_noisy=False)

    gold_cond_set = set([norm_text(x) for x in gold_cond_list if x])
    pred_cond_set = set([norm_text(x) for x in pred_cond_list if x])

    gold_med_set = set([norm_text(x) for x in gold_med_list if x])
    pred_med_set = set([norm_text(x) for x in pred_med_list if x])

    # Secondary exact metrics: source vs recon
    secondary_cond_exact = metrics_from_sets(gold_cond_set, pred_cond_set)
    secondary_med_exact = metrics_from_sets(gold_med_set, pred_med_set)

    # Secondary semantic metrics: source vs recon
    secondary_cond_sem = semantic_align_and_score(pid, "conditions", pred_cond_list, gold_cond_list)
    secondary_med_sem = semantic_align_and_score(pid, "medication_requests", pred_med_list, gold_med_list)

    # Official exact metrics: exact source items that are explicitly present in the note vs recon
    note_exact_cond_items = exact_items_supported_by_note(note_text, gold_cond_list)
    note_exact_med_items = exact_items_supported_by_note(note_text, gold_med_list)
    note_exact_cond_set = set([norm_text(x) for x in note_exact_cond_items if x])
    note_exact_med_set = set([norm_text(x) for x in note_exact_med_items if x])

    official_cond_exact = metrics_from_sets(note_exact_cond_set, pred_cond_set)
    official_med_exact = metrics_from_sets(note_exact_med_set, pred_med_set)

    # Official semantic metrics: note vs recon
    official_cond_sem = note_vs_recon_semantic_score(pid, "conditions", note_text, pred_cond_list)
    official_med_sem = note_vs_recon_semantic_score(pid, "medication_requests", note_text, pred_med_list)

    row = {
        "patient_id": pid,
        "note_style": note_style,
        "note_style_label": note_style_label,

        # Demographics (diagnostic only)
        "gold_full_name": gold.get("full_name"),
        "gold_birthDate": gold.get("birthDate"),
        "gold_gender": gold.get("gender"),
        "raw_patient_name": raw.get("patient_name"),
        "raw_date_of_birth": raw.get("date_of_birth"),
        "raw_gender": raw.get("gender"),
        "name_exact_raw": name_exact(gold.get("full_name") or "", raw.get("patient_name")),
        "dob_exact_raw": dob_exact(gold.get("birthDate"), raw.get("date_of_birth")),
        "gender_exact_raw": gender_exact(gold.get("gender"), raw.get("gender")),

        # Official = note vs recon : exact
        "score_official_note_vs_recon_conditions_exact_tp": int(official_cond_exact["tp"]),
        "score_official_note_vs_recon_conditions_exact_fp": int(official_cond_exact["fp"]),
        "score_official_note_vs_recon_conditions_exact_fn": int(official_cond_exact["fn"]),
        "score_official_note_vs_recon_conditions_exact_precision": float(official_cond_exact["precision"]),
        "score_official_note_vs_recon_conditions_exact_recall": float(official_cond_exact["recall"]),
        "score_official_note_vs_recon_conditions_exact_f1": float(official_cond_exact["f1"]),
        "score_official_note_vs_recon_medication_requests_exact_tp": int(official_med_exact["tp"]),
        "score_official_note_vs_recon_medication_requests_exact_fp": int(official_med_exact["fp"]),
        "score_official_note_vs_recon_medication_requests_exact_fn": int(official_med_exact["fn"]),
        "score_official_note_vs_recon_medication_requests_exact_precision": float(official_med_exact["precision"]),
        "score_official_note_vs_recon_medication_requests_exact_recall": float(official_med_exact["recall"]),
        "score_official_note_vs_recon_medication_requests_exact_f1": float(official_med_exact["f1"]),

        # Official = note vs recon : semantic
        "score_official_note_vs_recon_conditions_semantic_tp_llm": int(official_cond_sem["supported_n"]),
        "score_official_note_vs_recon_conditions_semantic_fp_llm": int(official_cond_sem["unsupported_n"]),
        "score_official_note_vs_recon_conditions_semantic_fn_llm": int(official_cond_sem["missing_n"]),
        "score_official_note_vs_recon_conditions_semantic_precision_llm": float(official_cond_sem["precision_note_consistency"]),
        "score_official_note_vs_recon_conditions_semantic_recall_llm": float(official_cond_sem["recall_note_consistency"]),
        "score_official_note_vs_recon_conditions_semantic_f1_llm": float(official_cond_sem["f1_note_consistency"]),
        "score_official_note_vs_recon_medication_requests_semantic_tp_llm": int(official_med_sem["supported_n"]),
        "score_official_note_vs_recon_medication_requests_semantic_fp_llm": int(official_med_sem["unsupported_n"]),
        "score_official_note_vs_recon_medication_requests_semantic_fn_llm": int(official_med_sem["missing_n"]),
        "score_official_note_vs_recon_medication_requests_semantic_precision_llm": float(official_med_sem["precision_note_consistency"]),
        "score_official_note_vs_recon_medication_requests_semantic_recall_llm": float(official_med_sem["recall_note_consistency"]),
        "score_official_note_vs_recon_medication_requests_semantic_f1_llm": float(official_med_sem["f1_note_consistency"]),

        # Secondary = source vs recon : exact
        "score_secondary_source_vs_recon_conditions_exact_tp": int(secondary_cond_exact["tp"]),
        "score_secondary_source_vs_recon_conditions_exact_fp": int(secondary_cond_exact["fp"]),
        "score_secondary_source_vs_recon_conditions_exact_fn": int(secondary_cond_exact["fn"]),
        "score_secondary_source_vs_recon_conditions_exact_precision": float(secondary_cond_exact["precision"]),
        "score_secondary_source_vs_recon_conditions_exact_recall": float(secondary_cond_exact["recall"]),
        "score_secondary_source_vs_recon_conditions_exact_f1": float(secondary_cond_exact["f1"]),
        "score_secondary_source_vs_recon_medication_requests_exact_tp": int(secondary_med_exact["tp"]),
        "score_secondary_source_vs_recon_medication_requests_exact_fp": int(secondary_med_exact["fp"]),
        "score_secondary_source_vs_recon_medication_requests_exact_fn": int(secondary_med_exact["fn"]),
        "score_secondary_source_vs_recon_medication_requests_exact_precision": float(secondary_med_exact["precision"]),
        "score_secondary_source_vs_recon_medication_requests_exact_recall": float(secondary_med_exact["recall"]),
        "score_secondary_source_vs_recon_medication_requests_exact_f1": float(secondary_med_exact["f1"]),

        # Secondary = source vs recon : semantic
        "score_secondary_source_vs_recon_conditions_semantic_tp_llm": int(secondary_cond_sem["tp"]),
        "score_secondary_source_vs_recon_conditions_semantic_fp_llm": int(secondary_cond_sem["fp"]),
        "score_secondary_source_vs_recon_conditions_semantic_fn_llm": int(secondary_cond_sem["fn"]),
        "score_secondary_source_vs_recon_conditions_semantic_precision_llm": float(secondary_cond_sem["precision_semantic"]),
        "score_secondary_source_vs_recon_conditions_semantic_recall_llm": float(secondary_cond_sem["recall_semantic"]),
        "score_secondary_source_vs_recon_conditions_semantic_f1_llm": float(secondary_cond_sem["f1_semantic"]),
        "score_secondary_source_vs_recon_medication_requests_semantic_tp_llm": int(secondary_med_sem["tp"]),
        "score_secondary_source_vs_recon_medication_requests_semantic_fp_llm": int(secondary_med_sem["fp"]),
        "score_secondary_source_vs_recon_medication_requests_semantic_fn_llm": int(secondary_med_sem["fn"]),
        "score_secondary_source_vs_recon_medication_requests_semantic_precision_llm": float(secondary_med_sem["precision_semantic"]),
        "score_secondary_source_vs_recon_medication_requests_semantic_recall_llm": float(secondary_med_sem["recall_semantic"]),
        "score_secondary_source_vs_recon_medication_requests_semantic_f1_llm": float(secondary_med_sem["f1_semantic"]),

        # Support counts and metadata
        "official_note_conditions_exact_source_items_detected_n": len(note_exact_cond_set),
        "official_note_medication_requests_exact_source_items_detected_n": len(note_exact_med_set),
        "gold_cond_n": len(gold_cond_list),
        "pred_cond_n": len(pred_cond_list),
        "raw_cond_n": len(ensure_list(raw.get("conditions"))),
        "gold_med_n": len(gold_med_list),
        "pred_med_n": len(pred_med_list),
        "raw_med_n": len(ensure_list(raw.get("medication_requests"))),

        # Legacy compatibility aliases
        "conditions_f1_string": float(secondary_cond_exact["f1"]),
        "medication_requests_f1_string": float(secondary_med_exact["f1"]),
        "conditions_f1_semantic_llm": float(secondary_cond_sem["f1_semantic"]),
        "medication_requests_f1_semantic_llm": float(secondary_med_sem["f1_semantic"]),
        "note_vs_recon_conditions_f1_llm": float(official_cond_sem["f1_note_consistency"]),
        "note_vs_recon_medication_requests_f1_llm": float(official_med_sem["f1_note_consistency"]),
        "note_vs_recon_conditions_precision_llm": float(official_cond_sem["precision_note_consistency"]),
        "note_vs_recon_medication_requests_precision_llm": float(official_med_sem["precision_note_consistency"]),
        "note_vs_recon_conditions_recall_llm": float(official_cond_sem["recall_note_consistency"]),
        "note_vs_recon_medication_requests_recall_llm": float(official_med_sem["recall_note_consistency"]),
        "note_vs_recon_conditions_supported_n_llm": int(official_cond_sem["supported_n"]),
        "note_vs_recon_medication_requests_supported_n_llm": int(official_med_sem["supported_n"]),
        "note_vs_recon_conditions_unsupported_n_llm": int(official_cond_sem["unsupported_n"]),
        "note_vs_recon_medication_requests_unsupported_n_llm": int(official_med_sem["unsupported_n"]),
        "note_vs_recon_conditions_missing_n_llm": int(official_cond_sem["missing_n"]),
        "note_vs_recon_medication_requests_missing_n_llm": int(official_med_sem["missing_n"]),
        "cond_fp_semantic": int(secondary_cond_sem["fp"]),
        "cond_fn_semantic": int(secondary_cond_sem["fn"]),
        "med_fp_semantic": int(secondary_med_sem["fp"]),
        "med_fn_semantic": int(secondary_med_sem["fn"]),
    }
    score_rows.append(row)

df_scores = pd.DataFrame(score_rows).sort_values(["note_style", "patient_id"]).reset_index(drop=True)

SCORES_DIR.mkdir(parents=True, exist_ok=True)
for r in score_rows:
    pid = r["patient_id"]
    (SCORES_DIR / f"{pid}.json").write_text(json.dumps(r, ensure_ascii=False, indent=2), encoding="utf-8")

(df_scores.sort_values("patient_id")
         .to_csv(SCORES_DIR / "df_scores.csv", index=False))

print("Scores dataset (official + secondary metrics, before error taxonomy)")
display(df_scores)

print("Mean name_exact_raw:", float(df_scores["name_exact_raw"].mean()))
print("Mean dob_exact_raw:", float(df_scores["dob_exact_raw"].mean()))
print("Mean gender_exact_raw:", float(df_scores["gender_exact_raw"].mean()))
print("Mean official note-vs-recon conditions exact_f1:", float(df_scores["score_official_note_vs_recon_conditions_exact_f1"].mean()))
print("Mean official note-vs-recon medication_requests exact_f1:", float(df_scores["score_official_note_vs_recon_medication_requests_exact_f1"].mean()))
print("Mean official note-vs-recon conditions semantic_f1_llm:", float(df_scores["score_official_note_vs_recon_conditions_semantic_f1_llm"].mean()))
print("Mean official note-vs-recon medication_requests semantic_f1_llm:", float(df_scores["score_official_note_vs_recon_medication_requests_semantic_f1_llm"].mean()))
print("Mean secondary source-vs-recon conditions exact_f1:", float(df_scores["score_secondary_source_vs_recon_conditions_exact_f1"].mean()))
print("Mean secondary source-vs-recon medication_requests exact_f1:", float(df_scores["score_secondary_source_vs_recon_medication_requests_exact_f1"].mean()))
print("Mean secondary source-vs-recon conditions semantic_f1_llm:", float(df_scores["score_secondary_source_vs_recon_conditions_semantic_f1_llm"].mean()))
print("Mean secondary source-vs-recon medication_requests semantic_f1_llm:", float(df_scores["score_secondary_source_vs_recon_medication_requests_semantic_f1_llm"].mean()))


Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

Scores dataset (official + secondary metrics, before error taxonomy)


,patient_id,note_style,note_style_label,gold_full_name,gold_birthDate,gold_gender,raw_patient_name,raw_date_of_birth,raw_gender,name_exact_raw,...,note_vs_recon_conditions_supported_n_llm,note_vs_recon_medication_requests_supported_n_llm,note_vs_recon_conditions_unsupported_n_llm,note_vs_recon_medication_requests_unsupported_n_llm,note_vs_recon_conditions_missing_n_llm,note_vs_recon_medication_requests_missing_n_llm,cond_fp_semantic,cond_fn_semantic,med_fp_semantic,med_fn_semantic
0,03777c32-ed98-50e2-f75d-cbcad532c610,consult_note_short,Short consultation note,Eloise59 Gerhold939,1968-03-26,female,Eloise59 Gerhold939,None,unknown,1,...,5,3,0,4,0,0,0,0,0,3
1,1ea40a05-c295-8b66-66a2-fda3bc1ed13f,consult_note_short,Short consultation note,Brock407 Paucek755,1960-05-25,male,Brock407 Paucek755,1960-05-25,male,1,...,2,3,0,1,0,0,0,2,0,0
2,51c7ff6a-33e3-3e1b-d3ad-0035c8227dfa,consult_note_short,Short consultation note,Georgiann138 Heaney114,2002-04-11,female,Georgiann138 Heaney114,2002-04-11,female,1,...,3,8,0,0,0,0,0,2,0,0
3,752a3c69-44d7-d6af-0340-7d3e748f6060,consult_note_short,Short consultation note,Oralia106 Williamson769,2000-08-10,female,Oralia Williamson,1999-08-08,female,0,...,2,2,0,2,0,0,1,0,0,0
4,7d28d76a-9ac8-67b4-3c88-0a75be3d0851,consult_note_short,Short consultation note,Erna640 Robel940,1969-01-03,female,Erna640 Robel940,1969-01-03,female,1,...,3,5,0,3,0,0,0,5,0,5
5,84fee56a-9ef7-8cff-e7d3-526ece562d87,consult_note_short,Short consultation note,Dewitt635 Bernier607,1953-09-21,male,Dewitt635 Bernier607,None,None,1,...,4,2,0,3,0,0,0,3,0,1
6,8c6ae452-5f8c-9ff6-006d-c6c860acf5cd,consult_note_short,Short consultation note,Darla154 Hoppe518,1917-05-15,female,Darla154 Hoppe518,1917-05-15,female,1,...,2,3,0,0,3,0,1,1,0,12
7,ce1153cb-d4ad-77e2-cd07-575e249a83ad,consult_note_short,Short consultation note,Deidra675 Rosenbaum794,2006-10-18,female,Deidra675 Rosenbaum794,2006-10-18,female,1,...,5,4,0,0,0,0,0,4,0,5
8,dc6c06d0-a7d8-100f-c08b-46c93700c188,consult_note_short,Short consultation note,Taylor21 Gulgowski816,2006-07-11,male,Taylor21 Gulgowski816,2006-07-11,male,1,...,1,2,0,0,1,0,0,0,0,0
9,fabb7ac2-1c62-3293-f43c-e35fb903c1c7,consult_note_short,Short consultation note,Nicholas495 Muller251,1988-08-29,male,Nicholas495 Muller251,1988-08-29,male,1,...,3,3,0,0,0,0,0,1,0,1


Mean name_exact_raw: 0.9
Mean dob_exact_raw: 0.85
Mean gender_exact_raw: 0.925
Mean official note-vs-recon conditions exact_f1: 0.39237720612720606
Mean official note-vs-recon medication_requests exact_f1: 0.8335004456327987
Mean official note-vs-recon conditions semantic_f1_llm: 0.9273638642059694
Mean official note-vs-recon medication_requests semantic_f1_llm: 0.8818603618603618
Mean secondary source-vs-recon conditions exact_f1: 0.26478797273604787
Mean secondary source-vs-recon medication_requests exact_f1: 0.8116102524926054
Mean secondary source-vs-recon conditions semantic_f1_llm: 0.8265187554047932
Mean secondary source-vs-recon medication_requests semantic_f1_llm: 0.9010676415088179


## 8) Chaîne de vérification LLM

Cette étape relit **`note_text`** et **`recon_clean`** pour produire un verdict patient par patient.

Objectif :
- signaler les reconstructions douteuses avant benchmark final ;
- quantifier les omissions et les hallucinations probables ;
- produire un dataset `df_verification` distinct des scores officiels.

Important :
- la vérification **n’altère pas** `recon_clean` ;
- elle ajoute une couche de confiance et d’interprétation.


In [16]:
# 8) LLM verification chain (note_clean + recon_clean -> verification verdict)

VERIFICATION_SCHEMA = {
    "verdict": "consistent|minor_issues|major_issues|unknown",
    "confidence": "number between 0 and 1",
    "unsupported_conditions": ["string"],
    "missing_conditions_from_reconstruction": ["string"],
    "unsupported_medication_requests": ["string"],
    "missing_medication_requests_from_reconstruction": ["string"],
    "summary": "string",
}

def verification_prompt(note_text: str, recon_clean_obj: dict) -> str:
    payload = {
        "conditions": recon_clean_obj.get("conditions", []),
        "medication_requests": recon_clean_obj.get("medication_requests", []),
    }
    return f"""You are verifying whether a structured reconstruction is faithfully supported by a clinical note.

Task:
Review the NOTE and the reconstructed structured data.
Return EXACTLY ONE valid JSON object following the schema below.

Rules:
- Use only information explicitly stated in the note.
- Do not infer new diagnoses, medications, procedures, or context.
- unsupported_* = items present in the reconstruction but not clearly supported by the note.
- missing_*_from_reconstruction = items clearly present in the note but missing from the reconstruction.
- verdict:
  - consistent: no material issue
  - minor_issues: small mismatch but reconstruction mostly usable
  - major_issues: important mismatch or several unsupported/missing items
  - unknown: impossible to judge reliably
- Return JSON only.

Schema:
{json.dumps(VERIFICATION_SCHEMA, ensure_ascii=False, indent=2)}

Clinical note:
<note>
{note_text}
</note>

Structured reconstruction:
<reconstruction>
{json.dumps(payload, ensure_ascii=False, indent=2)}
</reconstruction>

JSON:"""

def verify_reconstruction(pid: str, note_text: str, recon_clean_obj: dict) -> dict:
    signature_payload = {
        "note_text": note_text or "",
        "recon": {
            "conditions": recon_clean_obj.get("conditions", []),
            "medication_requests": recon_clean_obj.get("medication_requests", []),
        },
    }
    signature = hashlib.sha256(json.dumps(signature_payload, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()

    cache_path = VERIFICATION_DIR / f"{pid}.json"
    if cache_path.exists():
        try:
            cached = json.loads(cache_path.read_text(encoding="utf-8"))
            if cached.get("signature") == signature:
                return cached
        except Exception:
            pass

    out = {
        "patient_id": pid,
        "signature": signature,
        "verdict": "unknown",
        "confidence": 0.0,
        "unsupported_conditions": [],
        "missing_conditions_from_reconstruction": [],
        "unsupported_medication_requests": [],
        "missing_medication_requests_from_reconstruction": [],
        "summary": "",
    }

    if medgemma.available():
        raw = medgemma.generate(
            verification_prompt(note_text, recon_clean_obj),
            max_new_tokens=VERIFICATION_MAX_NEW_TOKENS,
            temperature=VERIFICATION_TEMPERATURE,
            do_sample=VERIFICATION_DO_SAMPLE,
        )
        obj = parse_llm_json(raw, retries=1) or {}
        out["verdict"] = str(obj.get("verdict", "unknown")).strip() or "unknown"
        try:
            out["confidence"] = float(obj.get("confidence", 0.0))
        except Exception:
            out["confidence"] = 0.0
        for key in [
            "unsupported_conditions",
            "missing_conditions_from_reconstruction",
            "unsupported_medication_requests",
            "missing_medication_requests_from_reconstruction",
        ]:
            val = obj.get(key, [])
            out[key] = postprocess_list(
                val if isinstance(val, list) else [],
                100,
                filter_noisy=("condition" in key),
            )
        out["summary"] = str(obj.get("summary", "")).strip()

    cache_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
    return out

verification_rows = []
for pid in sorted(recon_clean.keys()):
    note_obj = notes.get(pid)
    note_text = (note_obj.note if note_obj else "") or ""
    recon_obj = recon_clean.get(pid, {}) or {}
    gold_obj = gold_by_pid.get(pid, {}) or {}
    review = verify_reconstruction(pid, note_text, recon_obj)

    unsupported_total = len(review["unsupported_conditions"]) + len(review["unsupported_medication_requests"])
    missing_total = len(review["missing_conditions_from_reconstruction"]) + len(review["missing_medication_requests_from_reconstruction"])
    issue_total = unsupported_total + missing_total

    verification_rows.append({
        "patient_id": pid,
        "note_style": gold_obj.get("note_style"),
        "note_style_label": gold_obj.get("note_style_label"),
        "verification_verdict": review["verdict"],
        "verification_confidence": float(review["confidence"]),
        "verification_unsupported_conditions_n": len(review["unsupported_conditions"]),
        "verification_missing_conditions_n": len(review["missing_conditions_from_reconstruction"]),
        "verification_unsupported_medication_requests_n": len(review["unsupported_medication_requests"]),
        "verification_missing_medication_requests_n": len(review["missing_medication_requests_from_reconstruction"]),
        "verification_unsupported_total_n": unsupported_total,
        "verification_missing_total_n": missing_total,
        "verification_issue_total_n": issue_total,
        "verification_summary": review["summary"],
    })

df_verification = pd.DataFrame(verification_rows).sort_values(["note_style", "patient_id"]).reset_index(drop=True)
df_verification.to_csv(VERIFICATION_DIR / "df_verification.csv", index=False)

# Enrich df_scores with verification fields for downstream analysis/export
df_scores = df_scores.merge(df_verification, on=["patient_id", "note_style", "note_style_label"], how="left")

print("Verification dataset")
display(df_verification)

if {"note_style", "verification_issue_total_n", "verification_verdict"}.issubset(df_verification.columns):
    df_verification_by_note_style = (
        df_verification.groupby(["note_style", "note_style_label", "verification_verdict"], dropna=False)
        .agg(
            n_patients=("patient_id", "count"),
            mean_issue_total_n=("verification_issue_total_n", "mean"),
            mean_confidence=("verification_confidence", "mean"),
        )
        .reset_index()
        .sort_values(["note_style", "note_style_label", "n_patients"], ascending=[True, True, False])
    )
    df_verification_by_note_style.to_csv(VERIFICATION_DIR / "df_verification_by_note_style.csv", index=False)
    print("Verification summary by note_style")
    display(df_verification_by_note_style)
else:
    df_verification_by_note_style = pd.DataFrame()

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Verification dataset


,patient_id,note_style,note_style_label,verification_verdict,verification_confidence,verification_unsupported_conditions_n,verification_missing_conditions_n,verification_unsupported_medication_requests_n,verification_missing_medication_requests_n,verification_unsupported_total_n,verification_missing_total_n,verification_issue_total_n,verification_summary
0,03777c32-ed98-50e2-f75d-cbcad532c610,consult_note_short,Short consultation note,unknown,0.00,0,0,0,0,0,0,0,
1,1ea40a05-c295-8b66-66a2-fda3bc1ed13f,consult_note_short,Short consultation note,consistent,1.00,0,0,0,0,0,0,0,The reconstruction accurately reflects the inf...
2,51c7ff6a-33e3-3e1b-d3ad-0035c8227dfa,consult_note_short,Short consultation note,consistent,1.00,0,0,0,0,0,0,0,The reconstruction is consistent with the clin...
3,752a3c69-44d7-d6af-0340-7d3e748f6060,consult_note_short,Short consultation note,consistent,1.00,0,0,0,0,0,0,0,The reconstruction accurately reflects the inf...
4,7d28d76a-9ac8-67b4-3c88-0a75be3d0851,consult_note_short,Short consultation note,consistent,0.95,0,0,1,5,1,5,6,"The reconstruction is mostly consistent, but s..."
5,84fee56a-9ef7-8cff-e7d3-526ece562d87,consult_note_short,Short consultation note,consistent,1.00,0,0,3,1,3,1,4,The reconstruction is consistent with the clin...
6,8c6ae452-5f8c-9ff6-006d-c6c860acf5cd,consult_note_short,Short consultation note,consistent,1.00,0,0,2,1,2,1,3,The reconstruction accurately reflects the con...
7,ce1153cb-d4ad-77e2-cd07-575e249a83ad,consult_note_short,Short consultation note,consistent,1.00,0,0,1,0,1,0,1,The reconstruction accurately reflects the con...
8,dc6c06d0-a7d8-100f-c08b-46c93700c188,consult_note_short,Short consultation note,consistent,1.00,0,0,0,0,0,0,0,The reconstruction accurately reflects the inf...
9,fabb7ac2-1c62-3293-f43c-e35fb903c1c7,consult_note_short,Short consultation note,consistent,1.00,0,0,0,0,0,0,0,The reconstruction accurately reflects the con...


Verification summary by note_style


,note_style,note_style_label,verification_verdict,n_patients,mean_issue_total_n,mean_confidence
0,consult_note_short,Short consultation note,consistent,9,1.555556,0.994444
1,consult_note_short,Short consultation note,unknown,1,0.000000,0.000000
2,health_check_summary,Health check summary,consistent,9,0.666667,0.994444
3,health_check_summary,Health check summary,unknown,1,0.000000,0.000000
4,medical_history_note,Medical history note,consistent,9,0.666667,0.994444
5,medical_history_note,Medical history note,unknown,1,0.000000,0.000000
6,telegraphic_note,Telegraphic note,consistent,8,0.750000,1.000000
7,telegraphic_note,Telegraphic note,unknown,2,0.000000,0.000000


## 9) Analyse des erreurs, taxonomie et difficulté par `note_style`

Cette section sert à **interpréter** les scores.  
Les scores disent **combien** la reconstruction est bonne ou mauvaise.  
La taxonomie dit plutôt **pourquoi** elle est bonne ou mauvaise.

### Nomenclature de la taxonomie

Chaque patient reçoit, pour les **conditions** et pour les **medication requests**, quatre champs principaux :

- `overall` : catégorie globale de l’erreur observée
- `stage` : étape probable où le problème apparaît
- `severity` : niveau de gravité de l’erreur
- `explanation` : résumé textuel qui explique le diagnostic de taxonomie

### Sens des champs

**1. `overall`**  
C’est l’étiquette principale de lecture. Elle résume la nature du problème dominant.

Exemples de catégories utilisées :
- `perfect_reconstruction`
- `no_target_and_no_prediction`
- `complete_omission`
- `exact_wording_gap_only`
- `source_to_note_gap_or_generation_loss`
- `under_extraction_from_note`
- `over_extraction_from_note`
- `mixed_note_to_reconstruction_error`
- `canonicalization_or_exact_match_gap`
- `precision_dominant_pipeline_error`
- `recall_dominant_pipeline_error`
- `mixed_pipeline_error`
- `partial_pipeline_error`

**2. `stage`**  
Ce champ aide à localiser l’étape probable du problème.

Valeurs possibles :
- `none` : pas d’erreur significative
- `source_to_note` : la perte semble apparaître au moment de la génération de la note
- `note_to_reconstruction` : la note contient l’information, mais la reconstruction la récupère mal
- `canonicalization_exact_matching` : le problème vient surtout du wording exact ou de la canonicalisation
- `source_to_note_or_reconstruction` : cas ambigu de perte importante
- `mixed` : plusieurs causes sont plausibles en même temps

**3. `severity`**  
Niveau de gravité synthétique :
- `none`
- `mild`
- `moderate`
- `severe`

**4. `explanation`**  
Phrase compacte qui résume :
- la catégorie choisie
- l’étape probable
- les scores utiles
- les faux positifs / faux négatifs
- les omissions ou extractions en trop vis-à-vis de la note

### Lecture recommandée

Tu peux lire cette section de la manière suivante :

1. **Regarder les scores officiels**
   - `score_official_note_vs_recon_*`
2. **Regarder les scores secondaires**
   - `score_secondary_source_vs_recon_*`
3. **Utiliser la taxonomie**
   - pour comprendre si le problème vient surtout de la génération de note, de la reconstruction, ou seulement du matching exact

En résumé :

- **scores** = performance quantitative
- **taxonomie** = interprétation qualitative

### Couche supplémentaire ajoutée dans cette version

En plus de la taxonomie générale, cette section ajoute une lecture **par `note_style`** :

- distribution des catégories d’erreurs par style de note
- comparaison des styles les plus difficiles
- tableau des **hard cases** par style

L’objectif est de répondre à la question :

> quel type de note dégrade le plus la reconstruction, et sous quelle forme ?

In [17]:
# 9) Error taxonomy (interpretation layer on top of the scores)

def classify_error_taxonomy(
    gold_n: int,
    pred_n: int,
    f1_exact: float,
    f1_semantic: float,
    note_f1: float,
    note_precision: float,
    note_recall: float,
    fp_semantic: int,
    fn_semantic: int,
    note_unsupported_n: int,
    note_missing_n: int,
) -> dict:
    if gold_n == 0 and pred_n == 0:
        overall = "no_target_and_no_prediction"
        stage = "none"
    elif gold_n > 0 and pred_n == 0:
        overall = "complete_omission"
        stage = "note_to_reconstruction" if note_missing_n > 0 else "source_to_note_or_reconstruction"
    elif f1_exact == 1.0 and f1_semantic == 1.0 and note_f1 == 1.0:
        overall = "perfect_reconstruction"
        stage = "none"
    elif f1_exact < 1.0 and f1_semantic == 1.0 and note_f1 == 1.0:
        overall = "exact_wording_gap_only"
        stage = "canonicalization_exact_matching"
    elif note_f1 >= 0.9 and f1_semantic < 0.9:
        overall = "source_to_note_gap_or_generation_loss"
        stage = "source_to_note"
    elif f1_semantic >= 0.9 and note_f1 < 0.9:
        if note_missing_n > 0 and note_unsupported_n == 0:
            overall = "under_extraction_from_note"
        elif note_unsupported_n > 0 and note_missing_n == 0:
            overall = "over_extraction_from_note"
        else:
            overall = "mixed_note_to_reconstruction_error"
        stage = "note_to_reconstruction"
    elif f1_semantic > f1_exact and note_f1 >= 0.9:
        overall = "canonicalization_or_exact_match_gap"
        stage = "canonicalization_exact_matching"
    else:
        if fp_semantic > 0 and fn_semantic == 0:
            overall = "precision_dominant_pipeline_error"
        elif fn_semantic > 0 and fp_semantic == 0:
            overall = "recall_dominant_pipeline_error"
        elif fp_semantic > 0 and fn_semantic > 0:
            overall = "mixed_pipeline_error"
        else:
            overall = "partial_pipeline_error"
        stage = "mixed"

    quality = min(f1_semantic, note_f1)
    if overall in {"perfect_reconstruction", "no_target_and_no_prediction"}:
        severity = "none"
    elif quality >= 0.85:
        severity = "mild"
    elif quality >= 0.55:
        severity = "moderate"
    else:
        severity = "severe"

    explanation = (
        f"overall={overall}; stage={stage}; exact={f1_exact:.2f}; "
        f"semantic={f1_semantic:.2f}; note_consistency={note_f1:.2f}; "
        f"unsupported_from_note={note_unsupported_n}; missing_from_note={note_missing_n}; "
        f"fp_semantic={fp_semantic}; fn_semantic={fn_semantic}"
    )
    return {
        "overall": overall,
        "stage": stage,
        "severity": severity,
        "explanation": explanation,
    }

taxonomy_rows = []
for _, row in df_scores.iterrows():
    cond_tax = classify_error_taxonomy(
        gold_n=int(row["gold_cond_n"]),
        pred_n=int(row["pred_cond_n"]),
        f1_exact=float(row["score_secondary_source_vs_recon_conditions_exact_f1"]),
        f1_semantic=float(row["score_secondary_source_vs_recon_conditions_semantic_f1_llm"]),
        note_f1=float(row["score_official_note_vs_recon_conditions_semantic_f1_llm"]),
        note_precision=float(row["score_official_note_vs_recon_conditions_semantic_precision_llm"]),
        note_recall=float(row["score_official_note_vs_recon_conditions_semantic_recall_llm"]),
        fp_semantic=int(row["cond_fp_semantic"]),
        fn_semantic=int(row["cond_fn_semantic"]),
        note_unsupported_n=int(row["note_vs_recon_conditions_unsupported_n_llm"]),
        note_missing_n=int(row["note_vs_recon_conditions_missing_n_llm"]),
    )

    med_tax = classify_error_taxonomy(
        gold_n=int(row["gold_med_n"]),
        pred_n=int(row["pred_med_n"]),
        f1_exact=float(row["score_secondary_source_vs_recon_medication_requests_exact_f1"]),
        f1_semantic=float(row["score_secondary_source_vs_recon_medication_requests_semantic_f1_llm"]),
        note_f1=float(row["score_official_note_vs_recon_medication_requests_semantic_f1_llm"]),
        note_precision=float(row["score_official_note_vs_recon_medication_requests_semantic_precision_llm"]),
        note_recall=float(row["score_official_note_vs_recon_medication_requests_semantic_recall_llm"]),
        fp_semantic=int(row["med_fp_semantic"]),
        fn_semantic=int(row["med_fn_semantic"]),
        note_unsupported_n=int(row["note_vs_recon_medication_requests_unsupported_n_llm"]),
        note_missing_n=int(row["note_vs_recon_medication_requests_missing_n_llm"]),
    )

    pipeline_summary = (
        cond_tax["overall"]
        if cond_tax["overall"] == med_tax["overall"] and cond_tax["stage"] == med_tax["stage"]
        else f"conditions={cond_tax['overall']} | medication_requests={med_tax['overall']}"
    )

    taxonomy_rows.append({
        "patient_id": row["patient_id"],
        "error_taxonomy_conditions_overall": cond_tax["overall"],
        "error_taxonomy_conditions_stage": cond_tax["stage"],
        "error_taxonomy_conditions_severity": cond_tax["severity"],
        "error_taxonomy_conditions_explanation": cond_tax["explanation"],
        "error_taxonomy_medication_requests_overall": med_tax["overall"],
        "error_taxonomy_medication_requests_stage": med_tax["stage"],
        "error_taxonomy_medication_requests_severity": med_tax["severity"],
        "error_taxonomy_medication_requests_explanation": med_tax["explanation"],
        "error_taxonomy_pipeline_summary": pipeline_summary,
    })

df_error_taxonomy = pd.DataFrame(taxonomy_rows).sort_values("patient_id").reset_index(drop=True)

# Enrich the main score dataframe with taxonomy columns for easier downstream export/readability
df_scores = df_scores.merge(df_error_taxonomy, on="patient_id", how="left")

SCORES_DIR.mkdir(parents=True, exist_ok=True)
(df_error_taxonomy.sort_values("patient_id")
    .to_csv(SCORES_DIR / "df_error_taxonomy.csv", index=False))
(df_scores.sort_values("patient_id")
    .to_csv(SCORES_DIR / "df_scores.csv", index=False))

# Update per-patient score JSON files with taxonomy fields too
for _, r in df_scores.iterrows():
    pid = r["patient_id"]
    payload = {k: (None if pd.isna(v) else v) for k, v in r.to_dict().items()}
    (SCORES_DIR / f"{pid}.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

print("Error taxonomy dataset")
display(df_error_taxonomy)

print("Conditions taxonomy counts:")
display(df_error_taxonomy["error_taxonomy_conditions_overall"].value_counts(dropna=False).to_frame("n"))

print("Medication requests taxonomy counts:")
display(df_error_taxonomy["error_taxonomy_medication_requests_overall"].value_counts(dropna=False).to_frame("n"))
print("Taxonomy by note_style - conditions")
if {"note_style", "note_style_label", "error_taxonomy_conditions_overall"}.issubset(df_scores.columns):
    df_error_taxonomy_by_note_style_conditions = (
        df_scores.groupby(
            ["note_style", "note_style_label", "error_taxonomy_conditions_overall"],
            dropna=False
        )
        .size()
        .reset_index(name="n")
        .sort_values(["note_style", "note_style_label", "n"], ascending=[True, True, False])
        .reset_index(drop=True)
    )
    display(df_error_taxonomy_by_note_style_conditions)
else:
    df_error_taxonomy_by_note_style_conditions = pd.DataFrame()

print("Taxonomy by note_style - medication requests")
if {"note_style", "note_style_label", "error_taxonomy_medication_requests_overall"}.issubset(df_scores.columns):
    df_error_taxonomy_by_note_style_medication_requests = (
        df_scores.groupby(
            ["note_style", "note_style_label", "error_taxonomy_medication_requests_overall"],
            dropna=False
        )
        .size()
        .reset_index(name="n")
        .sort_values(["note_style", "note_style_label", "n"], ascending=[True, True, False])
        .reset_index(drop=True)
    )
    display(df_error_taxonomy_by_note_style_medication_requests)
else:
    df_error_taxonomy_by_note_style_medication_requests = pd.DataFrame()

difficulty_cols = [
    "score_secondary_source_vs_recon_conditions_semantic_f1_llm",
    "score_secondary_source_vs_recon_medication_requests_semantic_f1_llm",
    "score_official_note_vs_recon_conditions_semantic_f1_llm",
    "score_official_note_vs_recon_medication_requests_semantic_f1_llm",
]
difficulty_cols = [c for c in difficulty_cols if c in df_scores.columns]

if difficulty_cols:
    df_scores["difficulty_score"] = df_scores[difficulty_cols].mean(axis=1)
else:
    df_scores["difficulty_score"] = None

print("Hardest patients by note_style")
if {"note_style", "note_style_label", "patient_id", "difficulty_score"}.issubset(df_scores.columns):
    df_hard_cases_by_note_style = (
        df_scores.sort_values(["note_style", "difficulty_score", "patient_id"], ascending=[True, True, True])
        .groupby(["note_style", "note_style_label"], dropna=False)
        .head(3)
        [[
            "note_style", "note_style_label", "patient_id",
            "difficulty_score",
            "score_secondary_source_vs_recon_conditions_semantic_f1_llm",
            "score_secondary_source_vs_recon_medication_requests_semantic_f1_llm",
            "score_official_note_vs_recon_conditions_semantic_f1_llm",
            "score_official_note_vs_recon_medication_requests_semantic_f1_llm",
            "error_taxonomy_conditions_overall",
            "error_taxonomy_medication_requests_overall",
            "error_taxonomy_pipeline_summary",
        ]]
        .reset_index(drop=True)
    )
    display(df_hard_cases_by_note_style)
else:
    df_hard_cases_by_note_style = pd.DataFrame()

(df_error_taxonomy_by_note_style_conditions
    .to_csv(SCORES_DIR / "df_error_taxonomy_by_note_style_conditions.csv", index=False)) if not df_error_taxonomy_by_note_style_conditions.empty else None
(df_error_taxonomy_by_note_style_medication_requests
    .to_csv(SCORES_DIR / "df_error_taxonomy_by_note_style_medication_requests.csv", index=False)) if not df_error_taxonomy_by_note_style_medication_requests.empty else None
(df_hard_cases_by_note_style
    .to_csv(SCORES_DIR / "df_hard_cases_by_note_style.csv", index=False)) if not df_hard_cases_by_note_style.empty else None


Error taxonomy dataset


,patient_id,error_taxonomy_conditions_overall,error_taxonomy_conditions_stage,error_taxonomy_conditions_severity,error_taxonomy_conditions_explanation,error_taxonomy_medication_requests_overall,error_taxonomy_medication_requests_stage,error_taxonomy_medication_requests_severity,error_taxonomy_medication_requests_explanation,error_taxonomy_pipeline_summary
0,01f8bbfd-cfc6-3b97-8bc1-8da6f0b4a9a8,canonicalization_or_exact_match_gap,canonicalization_exact_matching,mild,overall=canonicalization_or_exact_match_gap; s...,over_extraction_from_note,note_to_reconstruction,moderate,overall=over_extraction_from_note; stage=note_...,conditions=canonicalization_or_exact_match_gap...
1,03777c32-ed98-50e2-f75d-cbcad532c610,exact_wording_gap_only,canonicalization_exact_matching,mild,overall=exact_wording_gap_only; stage=canonica...,recall_dominant_pipeline_error,mixed,moderate,overall=recall_dominant_pipeline_error; stage=...,conditions=exact_wording_gap_only | medication...
2,0a168e32-7b62-8597-0e11-296871bb764f,source_to_note_gap_or_generation_loss,source_to_note,moderate,overall=source_to_note_gap_or_generation_loss;...,perfect_reconstruction,none,none,overall=perfect_reconstruction; stage=none; ex...,conditions=source_to_note_gap_or_generation_lo...
3,122b4063-18dc-f20f-204c-6f2da758717b,exact_wording_gap_only,canonicalization_exact_matching,mild,overall=exact_wording_gap_only; stage=canonica...,perfect_reconstruction,none,none,overall=perfect_reconstruction; stage=none; ex...,conditions=exact_wording_gap_only | medication...
4,1293efbb-72dc-8b15-3bd6-916f96ed54f4,canonicalization_or_exact_match_gap,canonicalization_exact_matching,mild,overall=canonicalization_or_exact_match_gap; s...,perfect_reconstruction,none,none,overall=perfect_reconstruction; stage=none; ex...,conditions=canonicalization_or_exact_match_gap...
5,162135b4-931d-a531-c2ad-047b0c24abe8,exact_wording_gap_only,canonicalization_exact_matching,mild,overall=exact_wording_gap_only; stage=canonica...,source_to_note_gap_or_generation_loss,source_to_note,moderate,overall=source_to_note_gap_or_generation_loss;...,conditions=exact_wording_gap_only | medication...
6,1ea40a05-c295-8b66-66a2-fda3bc1ed13f,source_to_note_gap_or_generation_loss,source_to_note,moderate,overall=source_to_note_gap_or_generation_loss;...,over_extraction_from_note,note_to_reconstruction,mild,overall=over_extraction_from_note; stage=note_...,conditions=source_to_note_gap_or_generation_lo...
7,1f06f787-61cd-dd2d-a6b2-347507100f10,source_to_note_gap_or_generation_loss,source_to_note,moderate,overall=source_to_note_gap_or_generation_loss;...,perfect_reconstruction,none,none,overall=perfect_reconstruction; stage=none; ex...,conditions=source_to_note_gap_or_generation_lo...
8,246fb368-8991-dc93-f6a5-eca807e7dbde,canonicalization_or_exact_match_gap,canonicalization_exact_matching,mild,overall=canonicalization_or_exact_match_gap; s...,source_to_note_gap_or_generation_loss,source_to_note,mild,overall=source_to_note_gap_or_generation_loss;...,conditions=canonicalization_or_exact_match_gap...
9,355a3cad-a055-c14b-117e-47accfc708ca,source_to_note_gap_or_generation_loss,source_to_note,moderate,overall=source_to_note_gap_or_generation_loss;...,over_extraction_from_note,note_to_reconstruction,moderate,overall=over_extraction_from_note; stage=note_...,conditions=source_to_note_gap_or_generation_lo...


Conditions taxonomy counts:


,n
error_taxonomy_conditions_overall,
source_to_note_gap_or_generation_loss,17
exact_wording_gap_only,9
under_extraction_from_note,6
canonicalization_or_exact_match_gap,5
mixed_pipeline_error,1
complete_omission,1
recall_dominant_pipeline_error,1


Medication requests taxonomy counts:


...

,note_style,note_style_label,error_taxonomy_conditions_overall,n
0,consult_note_short,Short consultation note,source_to_note_gap_or_generation_loss,7
1,consult_note_short,Short consultation note,exact_wording_gap_only,1
2,consult_note_short,Short consultation note,mixed_pipeline_error,1
3,consult_note_short,Short consultation note,under_extraction_from_note,1
4,health_check_summary,Health check summary,exact_wording_gap_only,4
5,health_check_summary,Health check summary,source_to_note_gap_or_generation_loss,3
6,health_check_summary,Health check summary,canonicalization_or_exact_match_gap,2
7,health_check_summary,Health check summary,under_extraction_from_note,1
8,medical_history_note,Medical history note,source_to_note_gap_or_generation_loss,5
9,medical_history_note,Medical history note,under_extraction_from_note,3


Taxonomy by note_style - medication requests


,note_style,note_style_label,error_taxonomy_medication_requests_overall,n
0,consult_note_short,Short consultation note,over_extraction_from_note,3
1,consult_note_short,Short consultation note,source_to_note_gap_or_generation_loss,3
2,consult_note_short,Short consultation note,recall_dominant_pipeline_error,2
3,consult_note_short,Short consultation note,exact_wording_gap_only,1
4,consult_note_short,Short consultation note,perfect_reconstruction,1
5,health_check_summary,Health check summary,perfect_reconstruction,4
6,health_check_summary,Health check summary,canonicalization_or_exact_match_gap,2
7,health_check_summary,Health check summary,partial_pipeline_error,1
8,health_check_summary,Health check summary,recall_dominant_pipeline_error,1
9,health_check_summary,Health check summary,source_to_note_gap_or_generation_loss,1


Hardest patients by note_style


,note_style,note_style_label,patient_id,difficulty_score,score_secondary_source_vs_recon_conditions_semantic_f1_llm,score_secondary_source_vs_recon_medication_requests_semantic_f1_llm,score_official_note_vs_recon_conditions_semantic_f1_llm,score_official_note_vs_recon_medication_requests_semantic_f1_llm,error_taxonomy_conditions_overall,error_taxonomy_medication_requests_overall,error_taxonomy_pipeline_summary
0,consult_note_short,Short consultation note,8c6ae452-5f8c-9ff6-006d-c6c860acf5cd,0.601190,0.500000,0.333333,0.571429,1.000000,mixed_pipeline_error,source_to_note_gap_or_generation_loss,conditions=mixed_pipeline_error | medication_r...
1,consult_note_short,Short consultation note,7d28d76a-9ac8-67b4-3c88-0a75be3d0851,0.769148,0.545455,0.761905,1.000000,0.769231,source_to_note_gap_or_generation_loss,recall_dominant_pipeline_error,conditions=source_to_note_gap_or_generation_lo...
2,consult_note_short,Short consultation note,84fee56a-9ef7-8cff-e7d3-526ece562d87,0.801948,0.727273,0.909091,1.000000,0.571429,source_to_note_gap_or_generation_loss,over_extraction_from_note,conditions=source_to_note_gap_or_generation_lo...
3,health_check_summary,Health check summary,6f0b58f9-cb95-1fb3-5fef-cd914f9b4de3,0.803571,0.857143,0.857143,1.000000,0.500000,source_to_note_gap_or_generation_loss,recall_dominant_pipeline_error,conditions=source_to_note_gap_or_generation_lo...
4,health_check_summary,Health check summary,90053970-cbf7-9102-32f1-61c1b05827a8,0.833333,0.666667,1.000000,1.000000,0.666667,source_to_note_gap_or_generation_loss,under_extraction_from_note,conditions=source_to_note_gap_or_generation_lo...
5,health_check_summary,Health check summary,6ab08a28-1670-be6a-f2c9-f2e248700836,0.874472,0.714286,0.941176,0.909091,0.933333,source_to_note_gap_or_generation_loss,canonicalization_or_exact_match_gap,conditions=source_to_note_gap_or_generation_lo...
6,medical_history_note,Medical history note,965ecf4b-40d6-02e3-fe08-acd9eafc68fe,0.824519,0.875000,0.923077,1.000000,0.500000,source_to_note_gap_or_generation_loss,over_extraction_from_note,conditions=source_to_note_gap_or_generation_lo...
7,medical_history_note,Medical history note,9bbc77f6-c1e1-0c48-c64e-eb5958f15faf,0.833333,1.000000,1.000000,0.666667,0.666667,under_extraction_from_note,under_extraction_from_note,under_extraction_from_note
8,medical_history_note,Medical history note,3a364909-94ec-e0c4-0156-8a2e15fa354e,0.865385,0.461538,1.000000,1.000000,1.000000,source_to_note_gap_or_generation_loss,perfect_reconstruction,conditions=source_to_note_gap_or_generation_lo...
9,telegraphic_note,Telegraphic note,9b94051f-59a9-c941-61af-9acf2a9af22f,0.500000,0.000000,0.000000,1.000000,1.000000,complete_omission,complete_omission,complete_omission


## 9.1) Score summaries

Cette section calcule deux synthèses séparées :
- `df_scores_overall_summary` pour la vue globale du run
- `df_scores_summary_by_note_style` pour la comparaison statistique entre styles de notes

On garde ainsi une séparation claire :
- `df_scores` = niveau patient
- `df_scores_overall_summary` = vue globale
- `df_scores_summary_by_note_style` = vue statistique par `note_style`

In [18]:
# 9.1) Score summaries

SCORE_SUMMARY_METRIC_SPECS = [
    ("demographics", "demographics", "name_exact_raw"),
    ("demographics", "demographics", "dob_exact_raw"),
    ("demographics", "demographics", "gender_exact_raw"),

    ("official_note_vs_recon", "conditions", "score_official_note_vs_recon_conditions_exact_precision"),
    ("official_note_vs_recon", "conditions", "score_official_note_vs_recon_conditions_exact_recall"),
    ("official_note_vs_recon", "conditions", "score_official_note_vs_recon_conditions_exact_f1"),
    ("official_note_vs_recon", "conditions", "score_official_note_vs_recon_conditions_semantic_precision_llm"),
    ("official_note_vs_recon", "conditions", "score_official_note_vs_recon_conditions_semantic_recall_llm"),
    ("official_note_vs_recon", "conditions", "score_official_note_vs_recon_conditions_semantic_f1_llm"),

    ("official_note_vs_recon", "medication_requests", "score_official_note_vs_recon_medication_requests_exact_precision"),
    ("official_note_vs_recon", "medication_requests", "score_official_note_vs_recon_medication_requests_exact_recall"),
    ("official_note_vs_recon", "medication_requests", "score_official_note_vs_recon_medication_requests_exact_f1"),
    ("official_note_vs_recon", "medication_requests", "score_official_note_vs_recon_medication_requests_semantic_precision_llm"),
    ("official_note_vs_recon", "medication_requests", "score_official_note_vs_recon_medication_requests_semantic_recall_llm"),
    ("official_note_vs_recon", "medication_requests", "score_official_note_vs_recon_medication_requests_semantic_f1_llm"),

    ("secondary_source_vs_recon", "conditions", "score_secondary_source_vs_recon_conditions_exact_precision"),
    ("secondary_source_vs_recon", "conditions", "score_secondary_source_vs_recon_conditions_exact_recall"),
    ("secondary_source_vs_recon", "conditions", "score_secondary_source_vs_recon_conditions_exact_f1"),
    ("secondary_source_vs_recon", "conditions", "score_secondary_source_vs_recon_conditions_semantic_precision_llm"),
    ("secondary_source_vs_recon", "conditions", "score_secondary_source_vs_recon_conditions_semantic_recall_llm"),
    ("secondary_source_vs_recon", "conditions", "score_secondary_source_vs_recon_conditions_semantic_f1_llm"),

    ("secondary_source_vs_recon", "medication_requests", "score_secondary_source_vs_recon_medication_requests_exact_precision"),
    ("secondary_source_vs_recon", "medication_requests", "score_secondary_source_vs_recon_medication_requests_exact_recall"),
    ("secondary_source_vs_recon", "medication_requests", "score_secondary_source_vs_recon_medication_requests_exact_f1"),
    ("secondary_source_vs_recon", "medication_requests", "score_secondary_source_vs_recon_medication_requests_semantic_precision_llm"),
    ("secondary_source_vs_recon", "medication_requests", "score_secondary_source_vs_recon_medication_requests_semantic_recall_llm"),
    ("secondary_source_vs_recon", "medication_requests", "score_secondary_source_vs_recon_medication_requests_semantic_f1_llm"),
]

def build_scores_overall_summary(df_scores_in: pd.DataFrame, df_verification_in: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n_patients_total = int(len(df_scores_in))

    for metric_family, entity_type, metric_name in SCORE_SUMMARY_METRIC_SPECS:
        if metric_name not in df_scores_in.columns:
            continue
        series = pd.to_numeric(df_scores_in[metric_name], errors="coerce").dropna()
        if series.empty:
            continue
        rows.append({
            "summary_scope": "overall",
            "metric_family": metric_family,
            "entity_type": entity_type,
            "metric_name": metric_name,
            "mean_value": float(series.mean()),
            "median_value": float(series.median()),
            "std_value": float(series.std(ddof=0)) if len(series) > 1 else 0.0,
            "min_value": float(series.min()),
            "max_value": float(series.max()),
            "n_patients_total": n_patients_total,
            "n_non_null": int(series.notna().sum()),
            "aggregation_level": "patient_mean",
        })

    if df_verification_in is not None and not df_verification_in.empty:
        n_verif_total = int(len(df_verification_in))
        verdict_col = "verification_verdict" if "verification_verdict" in df_verification_in.columns else "verdict"
        for metric_name, verdict_value in [
            ("verification_consistent_rate", "consistent"),
            ("verification_minor_issues_rate", "minor_issues"),
            ("verification_major_issues_rate", "major_issues"),
            ("verification_unknown_rate", "unknown"),
        ]:
            series = (df_verification_in[verdict_col].fillna("") == verdict_value).astype(float)
            rows.append({
                "summary_scope": "overall",
                "metric_family": "verification",
                "entity_type": "all",
                "metric_name": metric_name,
                "mean_value": float(series.mean()) if len(series) else None,
                "median_value": float(series.median()) if len(series) else None,
                "std_value": float(series.std(ddof=0)) if len(series) > 1 else 0.0,
                "min_value": float(series.min()) if len(series) else None,
                "max_value": float(series.max()) if len(series) else None,
                "n_patients_total": n_verif_total,
                "n_non_null": int(series.notna().sum()),
                "aggregation_level": "patient_mean",
            })

        for metric_name, preferred_col, fallback_col in [
            ("verification_mean_confidence", "verification_confidence", "confidence"),
            ("verification_mean_issue_total_n", "verification_issue_total_n", "verification_issue_total_n"),
        ]:
            source_col = preferred_col if preferred_col in df_verification_in.columns else fallback_col
            if source_col not in df_verification_in.columns:
                continue
            series = pd.to_numeric(df_verification_in[source_col], errors="coerce").dropna()
            if series.empty:
                continue
            rows.append({
                "summary_scope": "overall",
                "metric_family": "verification",
                "entity_type": "all",
                "metric_name": metric_name,
                "mean_value": float(series.mean()),
                "median_value": float(series.median()),
                "std_value": float(series.std(ddof=0)) if len(series) > 1 else 0.0,
                "min_value": float(series.min()),
                "max_value": float(series.max()),
                "n_patients_total": n_verif_total,
                "n_non_null": int(series.notna().sum()),
                "aggregation_level": "patient_mean",
            })

    return pd.DataFrame(rows).sort_values(["metric_family", "entity_type", "metric_name"]).reset_index(drop=True)

def build_scores_summary_by_note_style(df_scores_in: pd.DataFrame, df_verification_in: pd.DataFrame) -> pd.DataFrame:
    rows = []

    if "note_style" in df_scores_in.columns:
        for (note_style, note_style_label), grp in df_scores_in.groupby(["note_style", "note_style_label"], dropna=False):
            n_patients_total = int(len(grp))
            for metric_family, entity_type, metric_name in SCORE_SUMMARY_METRIC_SPECS:
                if metric_name not in grp.columns:
                    continue
                series = pd.to_numeric(grp[metric_name], errors="coerce").dropna()
                if series.empty:
                    continue
                rows.append({
                    "summary_scope": "by_note_style",
                    "note_style": note_style,
                    "note_style_label": note_style_label,
                    "metric_family": metric_family,
                    "entity_type": entity_type,
                    "metric_name": metric_name,
                    "mean_value": float(series.mean()),
                    "median_value": float(series.median()),
                    "std_value": float(series.std(ddof=0)) if len(series) > 1 else 0.0,
                    "min_value": float(series.min()),
                    "max_value": float(series.max()),
                    "n_patients_total": n_patients_total,
                    "n_non_null": int(series.notna().sum()),
                    "aggregation_level": "patient_mean",
                })

    if df_verification_in is not None and not df_verification_in.empty and "note_style" in df_verification_in.columns:
        verdict_col = "verification_verdict" if "verification_verdict" in df_verification_in.columns else "verdict"
        for (note_style, note_style_label), grp in df_verification_in.groupby(["note_style", "note_style_label"], dropna=False):
            n_patients_total = int(len(grp))
            for metric_name, verdict_value in [
                ("verification_consistent_rate", "consistent"),
                ("verification_minor_issues_rate", "minor_issues"),
                ("verification_major_issues_rate", "major_issues"),
                ("verification_unknown_rate", "unknown"),
            ]:
                series = (grp[verdict_col].fillna("") == verdict_value).astype(float)
                rows.append({
                    "summary_scope": "by_note_style",
                    "note_style": note_style,
                    "note_style_label": note_style_label,
                    "metric_family": "verification",
                    "entity_type": "all",
                    "metric_name": metric_name,
                    "mean_value": float(series.mean()) if len(series) else None,
                    "median_value": float(series.median()) if len(series) else None,
                    "std_value": float(series.std(ddof=0)) if len(series) > 1 else 0.0,
                    "min_value": float(series.min()) if len(series) else None,
                    "max_value": float(series.max()) if len(series) else None,
                    "n_patients_total": n_patients_total,
                    "n_non_null": int(series.notna().sum()),
                    "aggregation_level": "patient_mean",
                })

            for metric_name, preferred_col, fallback_col in [
                ("verification_mean_confidence", "verification_confidence", "confidence"),
                ("verification_mean_issue_total_n", "verification_issue_total_n", "verification_issue_total_n"),
            ]:
                source_col = preferred_col if preferred_col in grp.columns else fallback_col
                if source_col not in grp.columns:
                    continue
                series = pd.to_numeric(grp[source_col], errors="coerce").dropna()
                if series.empty:
                    continue
                rows.append({
                    "summary_scope": "by_note_style",
                    "note_style": note_style,
                    "note_style_label": note_style_label,
                    "metric_family": "verification",
                    "entity_type": "all",
                    "metric_name": metric_name,
                    "mean_value": float(series.mean()),
                    "median_value": float(series.median()),
                    "std_value": float(series.std(ddof=0)) if len(series) > 1 else 0.0,
                    "min_value": float(series.min()),
                    "max_value": float(series.max()),
                    "n_patients_total": n_patients_total,
                    "n_non_null": int(series.notna().sum()),
                    "aggregation_level": "patient_mean",
                })

    summary = pd.DataFrame(rows)
    if summary.empty:
        return summary
    return summary.sort_values(["note_style", "metric_family", "entity_type", "metric_name"]).reset_index(drop=True)

df_scores_overall_summary = build_scores_overall_summary(df_scores, df_verification)
df_scores_summary_by_note_style = build_scores_summary_by_note_style(df_scores, df_verification)

if not df_scores_overall_summary.empty:
    df_scores_overall_summary.to_csv(SCORES_DIR / "df_scores_overall_summary.csv", index=False)
    print("Overall score summary")
    display(df_scores_overall_summary)
else:
    df_scores_overall_summary = pd.DataFrame()

if not df_scores_summary_by_note_style.empty:
    df_scores_summary_by_note_style.to_csv(SCORES_DIR / "df_scores_summary_by_note_style.csv", index=False)
    print("Score summary by note_style")
    display(df_scores_summary_by_note_style)
else:
    df_scores_summary_by_note_style = pd.DataFrame()

# Compatibility alias for older naming
df_scores_by_note_style = df_scores_summary_by_note_style.copy()

Overall score summary


,summary_scope,metric_family,entity_type,metric_name,mean_value,median_value,std_value,min_value,max_value,n_patients_total,n_non_null,aggregation_level
0,overall,demographics,demographics,dob_exact_raw,0.850000,1.000000,0.357071,0.000000,1.0,40,40,patient_mean
1,overall,demographics,demographics,gender_exact_raw,0.925000,1.000000,0.263391,0.000000,1.0,40,40,patient_mean
2,overall,demographics,demographics,name_exact_raw,0.900000,1.000000,0.300000,0.000000,1.0,40,40,patient_mean
3,overall,official_note_vs_recon,conditions,score_official_note_vs_recon_conditions_exact_f1,0.392377,0.444444,0.325637,0.000000,1.0,40,40,patient_mean
4,overall,official_note_vs_recon,conditions,score_official_note_vs_recon_conditions_exact_...,0.313823,0.300000,0.296105,0.000000,1.0,40,40,patient_mean
5,overall,official_note_vs_recon,conditions,score_official_note_vs_recon_conditions_exact_...,0.877083,1.000000,0.258862,0.000000,1.0,40,40,patient_mean
6,overall,official_note_vs_recon,conditions,score_official_note_vs_recon_conditions_semant...,0.927364,1.000000,0.137972,0.571429,1.0,40,40,patient_mean
7,overall,official_note_vs_recon,conditions,score_official_note_vs_recon_conditions_semant...,0.993333,1.000000,0.030000,0.833333,1.0,40,40,patient_mean
8,overall,official_note_vs_recon,conditions,score_official_note_vs_recon_conditions_semant...,0.897500,1.000000,0.205533,0.400000,1.0,40,40,patient_mean
9,overall,official_note_vs_recon,medication_requests,score_official_note_vs_recon_medication_reques...,0.833500,0.961538,0.225565,0.000000,1.0,40,40,patient_mean


Score summary by note_style


,summary_scope,note_style,note_style_label,metric_family,entity_type,metric_name,mean_value,median_value,std_value,min_value,max_value,n_patients_total,n_non_null,aggregation_level
0,by_note_style,consult_note_short,Short consultation note,demographics,demographics,dob_exact_raw,0.700000,1.0,0.458258,0.0,1.000000,10,10,patient_mean
1,by_note_style,consult_note_short,Short consultation note,demographics,demographics,gender_exact_raw,0.800000,1.0,0.400000,0.0,1.000000,10,10,patient_mean
2,by_note_style,consult_note_short,Short consultation note,demographics,demographics,name_exact_raw,0.900000,1.0,0.300000,0.0,1.000000,10,10,patient_mean
3,by_note_style,consult_note_short,Short consultation note,official_note_vs_recon,conditions,score_official_note_vs_recon_conditions_exact_f1,0.125714,0.0,0.202515,0.0,0.571429,10,10,patient_mean
4,by_note_style,consult_note_short,Short consultation note,official_note_vs_recon,conditions,score_official_note_vs_recon_conditions_exact_...,0.093333,0.0,0.149666,0.0,0.400000,10,10,patient_mean
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,by_note_style,telegraphic_note,Telegraphic note,verification,all,verification_major_issues_rate,0.000000,0.0,0.000000,0.0,0.000000,10,10,patient_mean
128,by_note_style,telegraphic_note,Telegraphic note,verification,all,verification_mean_confidence,0.800000,1.0,0.400000,0.0,1.000000,10,10,patient_mean
129,by_note_style,telegraphic_note,Telegraphic note,verification,all,verification_mean_issue_total_n,0.600000,0.0,1.496663,0.0,5.000000,10,10,patient_mean
130,by_note_style,telegraphic_note,Telegraphic note,verification,all,verification_minor_issues_rate,0.000000,0.0,0.000000,0.0,0.000000,10,10,patient_mean


## 10) Fichiers produits et lecture des sorties

- Dataset filtré: `synthea_local_simplified/filtered_conditions_medreq/`
- Gold: `data_bupa_conditions_medreq/gold/`
- Notes: `data_bupa_conditions_medreq/notes/`
- Reconstructions: `data_bupa_conditions_medreq/reconstruction/`
- Scores: `data_bupa_conditions_medreq/scores/`


## 10.1) Reset final benchmark outputs (optionnel mais recommandé avant le benchmark final)

Cette cellule supprime uniquement les sorties analytiques finales qui risquent de polluer le benchmark :
- `scores/`
- `verification/`
- `exports/`
- l’archive ZIP finale

Elle ne supprime pas le dataset filtré, le gold source, les notes ni la reconstruction.

In [19]:
# Reset final benchmark outputs (safe cleanup of final analytical artifacts only)

RESET_BENCHMARK_OUTPUTS = False  # Passe à True juste avant le benchmark final si tu veux repartir de sorties propres.

paths_to_clean = [
    SCORES_DIR,
    VERIFICATION_DIR,
    PIPE_DIR / "exports",
]
zip_to_remove = PIPE_DIR / "exports_datasets_full.zip"

def clear_dir_files(dir_path: Path, patterns: List[str]):
    if not dir_path.exists():
        return []
    removed = []
    for pattern in patterns:
        for p in sorted(dir_path.glob(pattern)):
            if p.is_file():
                p.unlink()
                removed.append(str(p))
    return removed

if RESET_BENCHMARK_OUTPUTS:
    removed_files = []
    for d in paths_to_clean:
        removed_files.extend(clear_dir_files(d, ["*.json", "*.csv"]))
    if zip_to_remove.exists():
        zip_to_remove.unlink()
        removed_files.append(str(zip_to_remove))

    print("Benchmark outputs reset complete.")
    print("Removed files:", len(removed_files))
    display(pd.DataFrame({"removed_file": removed_files}) if removed_files else pd.DataFrame(columns=["removed_file"]))
else:
    print("RESET_BENCHMARK_OUTPUTS = False -> no file removed.")
    print("Set it to True only when you want to clean scores / verification / exports before the final benchmark.")

RESET_BENCHMARK_OUTPUTS = False -> no file removed.
Set it to True only when you want to clean scores / verification / exports before the final benchmark.


## 11) Export des datasets depuis le cache

Cette section relit les sorties du cache, reconstruit tous les datasets finaux, les datasets de vérification et un `df_export_review` qui sert de relecture synthétique des exports.

In [20]:

# Export and inspect full datasets from on-disk cache (no truncated previews)
# Rebuilds df_index, df_gold, df_notes, df_recon, df_scores from cache folders
# note_text = cleaned note used for reconstruction
# raw_note_text = raw model output kept for debug only
# Main reconstruction = cleaned/no strict clipping
# df_scores contains official note-vs-recon metrics and secondary source-vs-recon metrics
# df_error_taxonomy is rebuilt and exported as a separate final dataset
# df_verification, df_scores_overall_summary, df_scores_summary_by_note_style and the export review are also exported here

from pathlib import Path
import json
import pandas as pd
import re
import zipfile

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

EXPORT_DIR = PIPE_DIR / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
RAW_NOTES_DIR = NOTES_DIR / "raw"
VERIFICATION_DIR = PIPE_DIR / "verification"

def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def list_json(dir_path: Path):
    if not dir_path.exists():
        return []
    return sorted([p for p in dir_path.glob("*.json") if p.is_file()])

def norm_text_local(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def f1_local(gold: set, pred: set) -> float:
    if not gold and not pred:
        return 1.0
    if not gold or not pred:
        return 0.0
    tp = len(gold & pred)
    fp = len(pred - gold)
    fn = len(gold - pred)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    return (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

def name_exact_local(gold_name: str, pred_name: str) -> int:
    return int(bool(norm_text_local(gold_name)) and norm_text_local(gold_name) == norm_text_local(pred_name or ""))

def dob_exact_local(gold_dob: str, pred_dob: str) -> int:
    g = (gold_dob or "").strip()
    p = (pred_dob or "").strip()
    return int(bool(g) and g == p)

def gender_exact_local(gold_gender: str, pred_gender: str) -> int:
    return int(bool(norm_text_local(gold_gender or "")) and norm_text_local(gold_gender or "") == norm_text_local(pred_gender or ""))

def shorten_local(text: str, n: int = 240) -> str:
    s = (text or "").replace("\n", " ").strip()
    return s[:n] + ("..." if len(s) > n else "")

def to_json_str(x) -> str:
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return ""

def ensure_list_local(x):
    return x if isinstance(x, list) else []

def zip_add_dir(z: zipfile.ZipFile, dir_path: Path, arc_prefix: str, pattern: str = "**/*"):
    if not dir_path.exists():
        return
    for p in sorted(dir_path.glob(pattern)):
        if p.is_file():
            rel = f"{arc_prefix}/{p.relative_to(dir_path).as_posix()}"
            z.write(p, arcname=rel)


# 9.1) Score summaries

SCORE_SUMMARY_METRIC_SPECS = [
    ("demographics", "demographics", "name_exact_raw"),
    ("demographics", "demographics", "dob_exact_raw"),
    ("demographics", "demographics", "gender_exact_raw"),

    ("official_note_vs_recon", "conditions", "score_official_note_vs_recon_conditions_exact_precision"),
    ("official_note_vs_recon", "conditions", "score_official_note_vs_recon_conditions_exact_recall"),
    ("official_note_vs_recon", "conditions", "score_official_note_vs_recon_conditions_exact_f1"),
    ("official_note_vs_recon", "conditions", "score_official_note_vs_recon_conditions_semantic_precision_llm"),
    ("official_note_vs_recon", "conditions", "score_official_note_vs_recon_conditions_semantic_recall_llm"),
    ("official_note_vs_recon", "conditions", "score_official_note_vs_recon_conditions_semantic_f1_llm"),

    ("official_note_vs_recon", "medication_requests", "score_official_note_vs_recon_medication_requests_exact_precision"),
    ("official_note_vs_recon", "medication_requests", "score_official_note_vs_recon_medication_requests_exact_recall"),
    ("official_note_vs_recon", "medication_requests", "score_official_note_vs_recon_medication_requests_exact_f1"),
    ("official_note_vs_recon", "medication_requests", "score_official_note_vs_recon_medication_requests_semantic_precision_llm"),
    ("official_note_vs_recon", "medication_requests", "score_official_note_vs_recon_medication_requests_semantic_recall_llm"),
    ("official_note_vs_recon", "medication_requests", "score_official_note_vs_recon_medication_requests_semantic_f1_llm"),

    ("secondary_source_vs_recon", "conditions", "score_secondary_source_vs_recon_conditions_exact_precision"),
    ("secondary_source_vs_recon", "conditions", "score_secondary_source_vs_recon_conditions_exact_recall"),
    ("secondary_source_vs_recon", "conditions", "score_secondary_source_vs_recon_conditions_exact_f1"),
    ("secondary_source_vs_recon", "conditions", "score_secondary_source_vs_recon_conditions_semantic_precision_llm"),
    ("secondary_source_vs_recon", "conditions", "score_secondary_source_vs_recon_conditions_semantic_recall_llm"),
    ("secondary_source_vs_recon", "conditions", "score_secondary_source_vs_recon_conditions_semantic_f1_llm"),

    ("secondary_source_vs_recon", "medication_requests", "score_secondary_source_vs_recon_medication_requests_exact_precision"),
    ("secondary_source_vs_recon", "medication_requests", "score_secondary_source_vs_recon_medication_requests_exact_recall"),
    ("secondary_source_vs_recon", "medication_requests", "score_secondary_source_vs_recon_medication_requests_exact_f1"),
    ("secondary_source_vs_recon", "medication_requests", "score_secondary_source_vs_recon_medication_requests_semantic_precision_llm"),
    ("secondary_source_vs_recon", "medication_requests", "score_secondary_source_vs_recon_medication_requests_semantic_recall_llm"),
    ("secondary_source_vs_recon", "medication_requests", "score_secondary_source_vs_recon_medication_requests_semantic_f1_llm"),
]

def build_scores_overall_summary(df_scores_in: pd.DataFrame, df_verification_in: pd.DataFrame) -> pd.DataFrame:
    rows = []
    n_patients_total = int(len(df_scores_in))

    for metric_family, entity_type, metric_name in SCORE_SUMMARY_METRIC_SPECS:
        if metric_name not in df_scores_in.columns:
            continue
        series = pd.to_numeric(df_scores_in[metric_name], errors="coerce").dropna()
        if series.empty:
            continue
        rows.append({
            "summary_scope": "overall",
            "metric_family": metric_family,
            "entity_type": entity_type,
            "metric_name": metric_name,
            "mean_value": float(series.mean()),
            "median_value": float(series.median()),
            "std_value": float(series.std(ddof=0)) if len(series) > 1 else 0.0,
            "min_value": float(series.min()),
            "max_value": float(series.max()),
            "n_patients_total": n_patients_total,
            "n_non_null": int(series.notna().sum()),
            "aggregation_level": "patient_mean",
        })

    if df_verification_in is not None and not df_verification_in.empty:
        n_verif_total = int(len(df_verification_in))
        verdict_col = "verification_verdict" if "verification_verdict" in df_verification_in.columns else "verdict"
        for metric_name, verdict_value in [
            ("verification_consistent_rate", "consistent"),
            ("verification_minor_issues_rate", "minor_issues"),
            ("verification_major_issues_rate", "major_issues"),
            ("verification_unknown_rate", "unknown"),
        ]:
            series = (df_verification_in[verdict_col].fillna("") == verdict_value).astype(float)
            rows.append({
                "summary_scope": "overall",
                "metric_family": "verification",
                "entity_type": "all",
                "metric_name": metric_name,
                "mean_value": float(series.mean()) if len(series) else None,
                "median_value": float(series.median()) if len(series) else None,
                "std_value": float(series.std(ddof=0)) if len(series) > 1 else 0.0,
                "min_value": float(series.min()) if len(series) else None,
                "max_value": float(series.max()) if len(series) else None,
                "n_patients_total": n_verif_total,
                "n_non_null": int(series.notna().sum()),
                "aggregation_level": "patient_mean",
            })

        for metric_name, preferred_col, fallback_col in [
            ("verification_mean_confidence", "verification_confidence", "confidence"),
            ("verification_mean_issue_total_n", "verification_issue_total_n", "verification_issue_total_n"),
        ]:
            source_col = preferred_col if preferred_col in df_verification_in.columns else fallback_col
            if source_col not in df_verification_in.columns:
                continue
            series = pd.to_numeric(df_verification_in[source_col], errors="coerce").dropna()
            if series.empty:
                continue
            rows.append({
                "summary_scope": "overall",
                "metric_family": "verification",
                "entity_type": "all",
                "metric_name": metric_name,
                "mean_value": float(series.mean()),
                "median_value": float(series.median()),
                "std_value": float(series.std(ddof=0)) if len(series) > 1 else 0.0,
                "min_value": float(series.min()),
                "max_value": float(series.max()),
                "n_patients_total": n_verif_total,
                "n_non_null": int(series.notna().sum()),
                "aggregation_level": "patient_mean",
            })

    return pd.DataFrame(rows).sort_values(["metric_family", "entity_type", "metric_name"]).reset_index(drop=True)

def build_scores_summary_by_note_style(df_scores_in: pd.DataFrame, df_verification_in: pd.DataFrame) -> pd.DataFrame:
    rows = []

    if "note_style" in df_scores_in.columns:
        for (note_style, note_style_label), grp in df_scores_in.groupby(["note_style", "note_style_label"], dropna=False):
            n_patients_total = int(len(grp))
            for metric_family, entity_type, metric_name in SCORE_SUMMARY_METRIC_SPECS:
                if metric_name not in grp.columns:
                    continue
                series = pd.to_numeric(grp[metric_name], errors="coerce").dropna()
                if series.empty:
                    continue
                rows.append({
                    "summary_scope": "by_note_style",
                    "note_style": note_style,
                    "note_style_label": note_style_label,
                    "metric_family": metric_family,
                    "entity_type": entity_type,
                    "metric_name": metric_name,
                    "mean_value": float(series.mean()),
                    "median_value": float(series.median()),
                    "std_value": float(series.std(ddof=0)) if len(series) > 1 else 0.0,
                    "min_value": float(series.min()),
                    "max_value": float(series.max()),
                    "n_patients_total": n_patients_total,
                    "n_non_null": int(series.notna().sum()),
                    "aggregation_level": "patient_mean",
                })

    if df_verification_in is not None and not df_verification_in.empty and "note_style" in df_verification_in.columns:
        verdict_col = "verification_verdict" if "verification_verdict" in df_verification_in.columns else "verdict"
        for (note_style, note_style_label), grp in df_verification_in.groupby(["note_style", "note_style_label"], dropna=False):
            n_patients_total = int(len(grp))
            for metric_name, verdict_value in [
                ("verification_consistent_rate", "consistent"),
                ("verification_minor_issues_rate", "minor_issues"),
                ("verification_major_issues_rate", "major_issues"),
                ("verification_unknown_rate", "unknown"),
            ]:
                series = (grp[verdict_col].fillna("") == verdict_value).astype(float)
                rows.append({
                    "summary_scope": "by_note_style",
                    "note_style": note_style,
                    "note_style_label": note_style_label,
                    "metric_family": "verification",
                    "entity_type": "all",
                    "metric_name": metric_name,
                    "mean_value": float(series.mean()) if len(series) else None,
                    "median_value": float(series.median()) if len(series) else None,
                    "std_value": float(series.std(ddof=0)) if len(series) > 1 else 0.0,
                    "min_value": float(series.min()) if len(series) else None,
                    "max_value": float(series.max()) if len(series) else None,
                    "n_patients_total": n_patients_total,
                    "n_non_null": int(series.notna().sum()),
                    "aggregation_level": "patient_mean",
                })

            for metric_name, preferred_col, fallback_col in [
                ("verification_mean_confidence", "verification_confidence", "confidence"),
                ("verification_mean_issue_total_n", "verification_issue_total_n", "verification_issue_total_n"),
            ]:
                source_col = preferred_col if preferred_col in grp.columns else fallback_col
                if source_col not in grp.columns:
                    continue
                series = pd.to_numeric(grp[source_col], errors="coerce").dropna()
                if series.empty:
                    continue
                rows.append({
                    "summary_scope": "by_note_style",
                    "note_style": note_style,
                    "note_style_label": note_style_label,
                    "metric_family": "verification",
                    "entity_type": "all",
                    "metric_name": metric_name,
                    "mean_value": float(series.mean()),
                    "median_value": float(series.median()),
                    "std_value": float(series.std(ddof=0)) if len(series) > 1 else 0.0,
                    "min_value": float(series.min()),
                    "max_value": float(series.max()),
                    "n_patients_total": n_patients_total,
                    "n_non_null": int(series.notna().sum()),
                    "aggregation_level": "patient_mean",
                })

    summary = pd.DataFrame(rows)
    if summary.empty:
        return summary
    return summary.sort_values(["note_style", "metric_family", "entity_type", "metric_name"]).reset_index(drop=True)

# --------
# 1) Index from filtered bundles (cache)
# --------
index_rows = []
for p in sorted(FILTERED_DIR.glob("*.json")) if FILTERED_DIR.exists() else []:
    try:
        b = read_json(p)
        patient = find_patient(b) or {}
        pid = patient.get("id") or p.stem
        n_cond = sum(1 for _ in iter_resources(b, "Condition"))
        n_medr = sum(1 for _ in iter_resources(b, "MedicationRequest"))
        index_rows.append({
            "patient_id": pid,
            "file_path": str(p),
            "n_conditions": n_cond,
            "n_medication_requests": n_medr,
        })
    except Exception:
        continue
df_index_cache = pd.DataFrame(index_rows).sort_values("patient_id").reset_index(drop=True)

# --------
# 2) Gold dataset (cache) with full lists
# --------
gold_rows = []
gold_by_pid_cache = {}

for p in list_json(GOLD_DIR):
    g = read_json(p)
    pid = g.get("patient_id") or p.stem
    gold_by_pid_cache[pid] = g

    conditions = ensure_list_local(g.get("conditions"))
    medreq = ensure_list_local(g.get("medication_requests"))

    gold_rows.append({
        "patient_id": pid,
        "full_name": g.get("full_name"),
        "birthDate": g.get("birthDate"),
        "gender": g.get("gender"),
        "note_style": g.get("note_style"),
        "note_style_label": g.get("note_style_label"),
        "n_conditions": len(conditions),
        "conditions": conditions,
        "conditions_json": to_json_str(conditions),
        "conditions_preview": "; ".join(conditions[:PREVIEW_N]),
        "n_medication_requests": len(medreq),
        "medication_requests": medreq,
        "medication_requests_json": to_json_str(medreq),
        "medreq_preview": "; ".join(medreq[:PREVIEW_N]),
        "gold_file": str(p),
    })
df_gold_cache = pd.DataFrame(gold_rows).sort_values(["note_style", "patient_id"]).reset_index(drop=True)

gold_cond_long = []
gold_med_long = []
for row in gold_rows:
    pid = row["patient_id"]
    for i, c in enumerate(row["conditions"]):
        gold_cond_long.append({"patient_id": pid, "rank": i, "condition": c})
    for i, m in enumerate(row["medication_requests"]):
        gold_med_long.append({"patient_id": pid, "rank": i, "medication_request": m})

df_gold_conditions_long = pd.DataFrame(gold_cond_long).sort_values(["patient_id", "rank"]).reset_index(drop=True)
df_gold_medreq_long = pd.DataFrame(gold_med_long).sort_values(["patient_id", "rank"]).reset_index(drop=True)

# --------
# 3) Notes dataset (cache) with clean note text + raw debug text
# --------
note_rows = []
for p in sorted(NOTES_DIR.glob("*.txt")) if NOTES_DIR.exists() else []:
    if p.parent.name == "raw":
        continue
    pid = p.stem
    note = p.read_text(encoding="utf-8")
    raw_path = RAW_NOTES_DIR / f"{pid}.txt"
    raw_note = raw_path.read_text(encoding="utf-8") if raw_path.exists() else note
    g = gold_by_pid_cache.get(pid, {}) or {}
    note_rows.append({
        "patient_id": pid,
        "full_name": g.get("full_name"),
        "birthDate": g.get("birthDate"),
        "gender": g.get("gender"),
        "note_style": g.get("note_style"),
        "note_style_label": g.get("note_style_label"),
        "raw_note_text": raw_note,
        "raw_note_preview": shorten_local(raw_note),
        "note_text": note,
        "note_preview": shorten_local(note),
        "raw_chars": len(raw_note or ""),
        "clean_chars": len(note or ""),
        "note_file": str(p),
        "raw_note_file": str(raw_path) if raw_path.exists() else "",
    })
df_notes_cache = pd.DataFrame(note_rows).sort_values(["note_style", "patient_id"]).reset_index(drop=True)

# --------
# 4) Reconstruction dataset (cache) with clean + clipped + raw views
# --------
recon_rows = []
recon_by_pid_cache = {}
recon_clipped_by_pid_cache = {}
recon_raw_by_pid_cache = {}

for p in list_json(RECON_DIR):
    if p.name.endswith(".raw.json") or p.name.endswith(".clipped.json"):
        continue

    pid = p.stem
    r = read_json(p)
    recon_by_pid_cache[pid] = r

    raw_path = RECON_DIR / f"{pid}.raw.json"
    raw_obj = read_json(raw_path) if raw_path.exists() else {}
    recon_raw_by_pid_cache[pid] = raw_obj if isinstance(raw_obj, dict) else {}

    clipped_path = RECON_DIR / f"{pid}.clipped.json"
    clipped_obj = read_json(clipped_path) if clipped_path.exists() else {}
    recon_clipped_by_pid_cache[pid] = clipped_obj if isinstance(clipped_obj, dict) else {}

    conditions = ensure_list_local(r.get("conditions"))
    medreq = ensure_list_local(r.get("medication_requests"))
    conditions_clipped = ensure_list_local(recon_clipped_by_pid_cache[pid].get("conditions"))
    medreq_clipped = ensure_list_local(recon_clipped_by_pid_cache[pid].get("medication_requests"))
    raw_conditions = ensure_list_local(recon_raw_by_pid_cache[pid].get("conditions"))
    raw_medreq = ensure_list_local(recon_raw_by_pid_cache[pid].get("medication_requests"))

    recon_rows.append({
        "patient_id": pid,
        "full_name": r.get("full_name"),
        "birthDate": r.get("birthDate"),
        "gender": r.get("gender"),
        "note_style": (gold_by_pid_cache.get(pid, {}) or {}).get("note_style"),
        "note_style_label": (gold_by_pid_cache.get(pid, {}) or {}).get("note_style_label"),

        "raw_conditions_n": len(raw_conditions),
        "raw_conditions": raw_conditions,
        "raw_conditions_json": to_json_str(raw_conditions),
        "raw_conditions_preview": "; ".join(raw_conditions[:PREVIEW_N]),

        "conditions_n": len(conditions),
        "conditions": conditions,
        "conditions_json": to_json_str(conditions),
        "conditions_preview": "; ".join(conditions[:PREVIEW_N]),

        "conditions_clipped_n": len(conditions_clipped),
        "conditions_clipped": conditions_clipped,
        "conditions_clipped_json": to_json_str(conditions_clipped),
        "conditions_clipped_preview": "; ".join(conditions_clipped[:PREVIEW_N]),

        "raw_medreq_n": len(raw_medreq),
        "raw_medication_requests": raw_medreq,
        "raw_medication_requests_json": to_json_str(raw_medreq),
        "raw_medreq_preview": "; ".join(raw_medreq[:PREVIEW_N]),

        "medreq_n": len(medreq),
        "medication_requests": medreq,
        "medication_requests_json": to_json_str(medreq),
        "medreq_preview": "; ".join(medreq[:PREVIEW_N]),

        "medreq_clipped_n": len(medreq_clipped),
        "medication_requests_clipped": medreq_clipped,
        "medication_requests_clipped_json": to_json_str(medreq_clipped),
        "medreq_clipped_preview": "; ".join(medreq_clipped[:PREVIEW_N]),

        "raw_patient_name": recon_raw_by_pid_cache[pid].get("patient_name"),
        "raw_date_of_birth": recon_raw_by_pid_cache[pid].get("date_of_birth"),
        "raw_gender": recon_raw_by_pid_cache[pid].get("gender"),

        "raw_file": str(raw_path) if raw_path.exists() else None,
        "recon_file": str(p),
        "recon_clipped_file": str(clipped_path) if clipped_path.exists() else None,
    })
df_recon_cache = pd.DataFrame(recon_rows).sort_values(["note_style", "patient_id"]).reset_index(drop=True)

recon_cond_long = []
recon_med_long = []
for row in recon_rows:
    pid = row["patient_id"]
    for i, c in enumerate(row["conditions"]):
        recon_cond_long.append({"patient_id": pid, "rank": i, "condition": c})
    for i, m in enumerate(row["medication_requests"]):
        recon_med_long.append({"patient_id": pid, "rank": i, "medication_request": m})

df_recon_conditions_long = pd.DataFrame(recon_cond_long).sort_values(["patient_id", "rank"]).reset_index(drop=True)
df_recon_medreq_long = pd.DataFrame(recon_med_long).sort_values(["patient_id", "rank"]).reset_index(drop=True)

# --------
# 5) Scores dataset
# --------
score_rows = []
score_files = list_json(SCORES_DIR)

if score_files:
    for p in score_files:
        if p.name == "df_scores.csv":
            continue
        try:
            obj = read_json(p)
            if isinstance(obj, dict) and "patient_id" in obj:
                score_rows.append(obj)
        except Exception:
            pass
else:
    pids = sorted(set(gold_by_pid_cache.keys()) | set(recon_by_pid_cache.keys()) | set(recon_raw_by_pid_cache.keys()))
    for pid in pids:
        g = gold_by_pid_cache.get(pid, {}) or {}
        r = recon_by_pid_cache.get(pid, {}) or {}
        raw = recon_raw_by_pid_cache.get(pid, {}) or {}
        r_clipped = recon_clipped_by_pid_cache.get(pid, {}) or {}

        gold_cond = set([norm_text_local(x) for x in ensure_list_local(g.get("conditions")) if x])
        pred_cond = set([norm_text_local(x) for x in ensure_list_local(r.get("conditions")) if x])
        secondary_cond_exact_tp = len(gold_cond & pred_cond)
        secondary_cond_exact_fp = len(pred_cond - gold_cond)
        secondary_cond_exact_fn = len(gold_cond - pred_cond)
        secondary_cond_exact_precision = secondary_cond_exact_tp / (secondary_cond_exact_tp + secondary_cond_exact_fp) if (secondary_cond_exact_tp + secondary_cond_exact_fp) else (1.0 if secondary_cond_exact_fn == 0 else 0.0)
        secondary_cond_exact_recall = secondary_cond_exact_tp / (secondary_cond_exact_tp + secondary_cond_exact_fn) if (secondary_cond_exact_tp + secondary_cond_exact_fn) else 1.0

        secondary_med_exact_tp = len(gold_med & pred_med)
        secondary_med_exact_fp = len(pred_med - gold_med)
        secondary_med_exact_fn = len(gold_med - pred_med)
        secondary_med_exact_precision = secondary_med_exact_tp / (secondary_med_exact_tp + secondary_med_exact_fp) if (secondary_med_exact_tp + secondary_med_exact_fp) else (1.0 if secondary_med_exact_fn == 0 else 0.0)
        secondary_med_exact_recall = secondary_med_exact_tp / (secondary_med_exact_tp + secondary_med_exact_fn) if (secondary_med_exact_tp + secondary_med_exact_fn) else 1.0

        score_rows.append({
            "patient_id": pid,
            "note_style": g.get("note_style"),
            "note_style_label": g.get("note_style_label"),
            "gold_full_name": g.get("full_name"),
            "gold_birthDate": g.get("birthDate"),
            "gold_gender": g.get("gender"),
            "raw_patient_name": raw.get("patient_name"),
            "raw_date_of_birth": raw.get("date_of_birth"),
            "raw_gender": raw.get("gender"),
            "name_exact_raw": name_exact_local(g.get("full_name") or "", raw.get("patient_name") or ""),
            "dob_exact_raw": dob_exact_local(g.get("birthDate") or "", raw.get("date_of_birth") or ""),
            "gender_exact_raw": gender_exact_local(g.get("gender") or "", raw.get("gender") or ""),

            "score_official_note_vs_recon_conditions_exact_tp": None,
            "score_official_note_vs_recon_conditions_exact_fp": None,
            "score_official_note_vs_recon_conditions_exact_fn": None,
            "score_official_note_vs_recon_conditions_exact_precision": None,
            "score_official_note_vs_recon_conditions_exact_recall": None,
            "score_official_note_vs_recon_conditions_exact_f1": None,
            "score_official_note_vs_recon_medication_requests_exact_tp": None,
            "score_official_note_vs_recon_medication_requests_exact_fp": None,
            "score_official_note_vs_recon_medication_requests_exact_fn": None,
            "score_official_note_vs_recon_medication_requests_exact_precision": None,
            "score_official_note_vs_recon_medication_requests_exact_recall": None,
            "score_official_note_vs_recon_medication_requests_exact_f1": None,

            "score_official_note_vs_recon_conditions_semantic_tp_llm": None,
            "score_official_note_vs_recon_conditions_semantic_fp_llm": None,
            "score_official_note_vs_recon_conditions_semantic_fn_llm": None,
            "score_official_note_vs_recon_conditions_semantic_precision_llm": None,
            "score_official_note_vs_recon_conditions_semantic_recall_llm": None,
            "score_official_note_vs_recon_conditions_semantic_f1_llm": None,
            "score_official_note_vs_recon_medication_requests_semantic_tp_llm": None,
            "score_official_note_vs_recon_medication_requests_semantic_fp_llm": None,
            "score_official_note_vs_recon_medication_requests_semantic_fn_llm": None,
            "score_official_note_vs_recon_medication_requests_semantic_precision_llm": None,
            "score_official_note_vs_recon_medication_requests_semantic_recall_llm": None,
            "score_official_note_vs_recon_medication_requests_semantic_f1_llm": None,

            "score_secondary_source_vs_recon_conditions_exact_tp": secondary_cond_exact_tp,
            "score_secondary_source_vs_recon_conditions_exact_fp": secondary_cond_exact_fp,
            "score_secondary_source_vs_recon_conditions_exact_fn": secondary_cond_exact_fn,
            "score_secondary_source_vs_recon_conditions_exact_precision": secondary_cond_exact_precision,
            "score_secondary_source_vs_recon_conditions_exact_recall": secondary_cond_exact_recall,
            "score_secondary_source_vs_recon_conditions_exact_f1": f1_local(gold_cond, pred_cond),
            "score_secondary_source_vs_recon_medication_requests_exact_tp": secondary_med_exact_tp,
            "score_secondary_source_vs_recon_medication_requests_exact_fp": secondary_med_exact_fp,
            "score_secondary_source_vs_recon_medication_requests_exact_fn": secondary_med_exact_fn,
            "score_secondary_source_vs_recon_medication_requests_exact_precision": secondary_med_exact_precision,
            "score_secondary_source_vs_recon_medication_requests_exact_recall": secondary_med_exact_recall,
            "score_secondary_source_vs_recon_medication_requests_exact_f1": f1_local(gold_med, pred_med),

            "score_secondary_source_vs_recon_conditions_semantic_tp_llm": None,
            "score_secondary_source_vs_recon_conditions_semantic_fp_llm": None,
            "score_secondary_source_vs_recon_conditions_semantic_fn_llm": None,
            "score_secondary_source_vs_recon_conditions_semantic_precision_llm": None,
            "score_secondary_source_vs_recon_conditions_semantic_recall_llm": None,
            "score_secondary_source_vs_recon_conditions_semantic_f1_llm": None,
            "score_secondary_source_vs_recon_medication_requests_semantic_tp_llm": None,
            "score_secondary_source_vs_recon_medication_requests_semantic_fp_llm": None,
            "score_secondary_source_vs_recon_medication_requests_semantic_fn_llm": None,
            "score_secondary_source_vs_recon_medication_requests_semantic_precision_llm": None,
            "score_secondary_source_vs_recon_medication_requests_semantic_recall_llm": None,
            "score_secondary_source_vs_recon_medication_requests_semantic_f1_llm": None,

            "conditions_f1_string": f1_local(gold_cond, pred_cond),
            "medication_requests_f1_string": f1_local(gold_med, pred_med),
            "conditions_f1_semantic_llm": None,
            "medication_requests_f1_semantic_llm": None,
            "note_vs_recon_conditions_f1_llm": None,
            "note_vs_recon_medication_requests_f1_llm": None,
            "note_vs_recon_conditions_precision_llm": None,
            "note_vs_recon_medication_requests_precision_llm": None,
            "note_vs_recon_conditions_recall_llm": None,
            "note_vs_recon_medication_requests_recall_llm": None,
            "note_vs_recon_conditions_supported_n_llm": None,
            "note_vs_recon_medication_requests_supported_n_llm": None,
            "note_vs_recon_conditions_unsupported_n_llm": None,
            "note_vs_recon_medication_requests_unsupported_n_llm": None,
            "note_vs_recon_conditions_missing_n_llm": None,
            "note_vs_recon_medication_requests_missing_n_llm": None,
            "cond_fp_semantic": None,
            "cond_fn_semantic": None,
            "med_fp_semantic": None,
            "med_fn_semantic": None,

            "error_taxonomy_conditions_overall": None,
            "error_taxonomy_conditions_stage": None,
            "error_taxonomy_conditions_severity": None,
            "error_taxonomy_conditions_explanation": None,
            "error_taxonomy_medication_requests_overall": None,
            "error_taxonomy_medication_requests_stage": None,
            "error_taxonomy_medication_requests_severity": None,
            "error_taxonomy_medication_requests_explanation": None,
            "error_taxonomy_pipeline_summary": None,
            "gold_cond_n": len(gold_cond),
            "pred_cond_n": len(pred_cond),
            "raw_cond_n": len(ensure_list_local(raw.get("conditions"))),
            "gold_med_n": len(gold_med),
            "pred_med_n": len(pred_med),
            "raw_med_n": len(ensure_list_local(raw.get("medication_requests"))),
        })

df_scores_cache = pd.DataFrame(score_rows).sort_values(["note_style", "patient_id"]).reset_index(drop=True)


verification_rows_cache = []
verification_files = list_json(VERIFICATION_DIR)
for p in verification_files:
    if p.name in {"df_verification.csv", "df_verification_by_note_style.csv"}:
        continue
    try:
        obj = read_json(p)
        if isinstance(obj, dict) and "patient_id" in obj:
            verification_rows_cache.append(obj)
    except Exception:
        pass

df_verification_cache = pd.DataFrame(verification_rows_cache)
if not df_verification_cache.empty:
    meta_cols = [c for c in ["patient_id", "note_style", "note_style_label"] if c in df_scores_cache.columns]
    if meta_cols:
        df_verification_cache = df_verification_cache.merge(
            df_scores_cache[meta_cols].drop_duplicates("patient_id"),
            on="patient_id",
            how="left"
        )
    df_verification_cache = df_verification_cache.sort_values(["note_style", "patient_id"]).reset_index(drop=True)

if not df_verification_cache.empty and {"note_style", "note_style_label", "verification_verdict"}.issubset(df_verification_cache.columns):
    df_verification_by_note_style_cache = (
        df_verification_cache.groupby(["note_style", "note_style_label", "verification_verdict"], dropna=False)
        .agg(
            n_patients=("patient_id", "count"),
            mean_issue_total_n=("verification_issue_total_n", "mean"),
            mean_confidence=("verification_confidence", "mean"),
        )
        .reset_index()
        .sort_values(["note_style", "note_style_label", "n_patients"], ascending=[True, True, False])
    )
else:
    df_verification_by_note_style_cache = pd.DataFrame()


taxonomy_cols_cache = [c for c in [
    "patient_id",
    "error_taxonomy_conditions_overall",
    "error_taxonomy_conditions_stage",
    "error_taxonomy_conditions_severity",
    "error_taxonomy_conditions_explanation",
    "error_taxonomy_medication_requests_overall",
    "error_taxonomy_medication_requests_stage",
    "error_taxonomy_medication_requests_severity",
    "error_taxonomy_medication_requests_explanation",
    "error_taxonomy_pipeline_summary",
] if c in df_scores_cache.columns]
df_error_taxonomy_cache = df_scores_cache[taxonomy_cols_cache].copy() if taxonomy_cols_cache else pd.DataFrame()

print("Index dataset (from filtered cache)")
display(df_index_cache.head(10))

df_generation_view_cache = (
    df_gold_cache[[
        "patient_id", "full_name", "birthDate", "gender",
        "note_style", "note_style_label",
        "conditions_preview", "medreq_preview"
    ]]
    .merge(
        df_notes_cache[[
            "patient_id",
            "used_llm", "model", "raw_note_preview", "note_preview",
            "raw_chars", "clean_chars"
        ]] if {"used_llm", "model", "raw_note_preview", "note_preview", "raw_chars", "clean_chars"}.issubset(df_notes_cache.columns)
        else df_notes_cache[["patient_id", "raw_note_preview", "note_preview", "raw_chars", "clean_chars"]],
        on="patient_id",
        how="left",
    )
    .sort_values(["note_style", "patient_id"])
    .reset_index(drop=True)
)

print("Generation dataset view (from cache) - gold source + generated notes")
display(df_generation_view_cache.head(10))

if "note_style" in df_generation_view_cache.columns:
    print("Note style distribution (from cache)")
    display(df_generation_view_cache["note_style"].value_counts(dropna=False).to_frame("n"))

print("Reconstruction dataset (from cache) - clean main view + clipped debug + raw extraction")
display(df_recon_cache.head(10))

print("Scores dataset (from cache) - main scores compare gold filtered source vs clean reconstruction, plus note_text vs clean reconstruction")
display(df_scores_cache.head(10))

if not df_verification_cache.empty:
    print("Verification dataset (from cache)")
    display(df_verification_cache.head(10))

if not df_verification_by_note_style_cache.empty:
    print("Verification summary by note_style (from cache)")
    display(df_verification_by_note_style_cache.head(10))



df_scores_overall_summary_cache = build_scores_overall_summary(df_scores_cache, df_verification_cache)
if not df_scores_overall_summary_cache.empty:
    print("Overall score summary (from cache)")
    display(df_scores_overall_summary_cache)

df_scores_summary_by_note_style_cache = build_scores_summary_by_note_style(df_scores_cache, df_verification_cache)
if not df_scores_summary_by_note_style_cache.empty:
    print("Score summary by note_style (from cache)")
    display(df_scores_summary_by_note_style_cache)

if {"note_style", "note_style_label", "error_taxonomy_conditions_overall"}.issubset(df_scores_cache.columns):
    df_error_taxonomy_by_note_style_conditions_cache = (
        df_scores_cache.groupby(
            ["note_style", "note_style_label", "error_taxonomy_conditions_overall"],
            dropna=False
        )
        .size()
        .reset_index(name="n")
        .sort_values(["note_style", "note_style_label", "n"], ascending=[True, True, False])
        .reset_index(drop=True)
    )
else:
    df_error_taxonomy_by_note_style_conditions_cache = pd.DataFrame()

if {"note_style", "note_style_label", "error_taxonomy_medication_requests_overall"}.issubset(df_scores_cache.columns):
    df_error_taxonomy_by_note_style_medication_requests_cache = (
        df_scores_cache.groupby(
            ["note_style", "note_style_label", "error_taxonomy_medication_requests_overall"],
            dropna=False
        )
        .size()
        .reset_index(name="n")
        .sort_values(["note_style", "note_style_label", "n"], ascending=[True, True, False])
        .reset_index(drop=True)
    )
else:
    df_error_taxonomy_by_note_style_medication_requests_cache = pd.DataFrame()

difficulty_cols_cache = [
    "score_secondary_source_vs_recon_conditions_semantic_f1_llm",
    "score_secondary_source_vs_recon_medication_requests_semantic_f1_llm",
    "score_official_note_vs_recon_conditions_semantic_f1_llm",
    "score_official_note_vs_recon_medication_requests_semantic_f1_llm",
]
difficulty_cols_cache = [c for c in difficulty_cols_cache if c in df_scores_cache.columns]
if difficulty_cols_cache:
    df_scores_cache["difficulty_score"] = df_scores_cache[difficulty_cols_cache].mean(axis=1)
else:
    df_scores_cache["difficulty_score"] = None

if {"note_style", "note_style_label", "patient_id", "difficulty_score"}.issubset(df_scores_cache.columns):
    df_hard_cases_by_note_style_cache = (
        df_scores_cache.sort_values(["note_style", "difficulty_score", "patient_id"], ascending=[True, True, True])
        .groupby(["note_style", "note_style_label"], dropna=False)
        .head(3)
        [[
            "note_style", "note_style_label", "patient_id",
            "difficulty_score",
            "score_secondary_source_vs_recon_conditions_semantic_f1_llm",
            "score_secondary_source_vs_recon_medication_requests_semantic_f1_llm",
            "score_official_note_vs_recon_conditions_semantic_f1_llm",
            "score_official_note_vs_recon_medication_requests_semantic_f1_llm",
            "error_taxonomy_conditions_overall",
            "error_taxonomy_medication_requests_overall",
            "error_taxonomy_pipeline_summary",
        ]]
        .reset_index(drop=True)
    )
else:
    df_hard_cases_by_note_style_cache = pd.DataFrame()

if not df_error_taxonomy_cache.empty:
    print("Error taxonomy dataset (from cache)")
    display(df_error_taxonomy_cache.head(10))
if not df_error_taxonomy_by_note_style_conditions_cache.empty:
    print("Error taxonomy by note_style - conditions (from cache)")
    display(df_error_taxonomy_by_note_style_conditions_cache.head(20))

if not df_error_taxonomy_by_note_style_medication_requests_cache.empty:
    print("Error taxonomy by note_style - medication requests (from cache)")
    display(df_error_taxonomy_by_note_style_medication_requests_cache.head(20))

if not df_hard_cases_by_note_style_cache.empty:
    print("Hardest patients by note_style (from cache)")
    display(df_hard_cases_by_note_style_cache)

print("\nTip: to inspect one patient in full, pick a patient_id and do:")
print("  pid = df_gold_cache.iloc[0]['patient_id']")
print("  df_gold_cache.loc[df_gold_cache.patient_id==pid, ['conditions','medication_requests']].iloc[0].to_dict()")
print("  df_recon_cache.loc[df_recon_cache.patient_id==pid, ['raw_conditions','conditions','conditions_clipped']].iloc[0].to_dict()")
print("  print(df_notes_cache.loc[df_notes_cache.patient_id==pid, 'note_text'].iloc[0])")

exports = {
    "df_index": df_index_cache,
    "df_gold": df_gold_cache,
    "df_notes": df_notes_cache,
    "df_recon": df_recon_cache,
    "df_scores": df_scores_cache,
    "df_gold_conditions_long": df_gold_conditions_long,
    "df_gold_medreq_long": df_gold_medreq_long,
    "df_recon_conditions_long": df_recon_conditions_long,
    "df_recon_medreq_long": df_recon_medreq_long,
}
if not df_error_taxonomy_cache.empty:
    exports["df_error_taxonomy"] = df_error_taxonomy_cache
if not df_scores_summary_by_note_style_cache.empty:
    exports["df_scores_summary_by_note_style"] = df_scores_summary_by_note_style_cache
if not df_scores_overall_summary_cache.empty:
    exports["df_scores_overall_summary"] = df_scores_overall_summary_cache
if not df_error_taxonomy_by_note_style_conditions_cache.empty:
    exports["df_error_taxonomy_by_note_style_conditions"] = df_error_taxonomy_by_note_style_conditions_cache
if not df_error_taxonomy_by_note_style_medication_requests_cache.empty:
    exports["df_error_taxonomy_by_note_style_medication_requests"] = df_error_taxonomy_by_note_style_medication_requests_cache
if not df_hard_cases_by_note_style_cache.empty:
    exports["df_hard_cases_by_note_style"] = df_hard_cases_by_note_style_cache
if not df_verification_cache.empty:
    exports["df_verification"] = df_verification_cache
if not df_verification_by_note_style_cache.empty:
    exports["df_verification_by_note_style"] = df_verification_by_note_style_cache

for name, df in exports.items():
    out_csv = EXPORT_DIR / f"{name}.csv"
    df.to_csv(out_csv, index=False)
    print("Saved:", out_csv)



# Export review
export_review_rows = []
for name, df in exports.items():
    export_review_rows.append({
        "dataset_name": name,
        "rows": int(len(df)),
        "cols": int(len(df.columns)),
        "csv_path": str(EXPORT_DIR / f"{name}.csv"),
    })
df_export_review = pd.DataFrame(export_review_rows).sort_values("dataset_name").reset_index(drop=True)
df_export_review.to_csv(EXPORT_DIR / "df_export_review.csv", index=False)
exports["df_export_review"] = df_export_review
print("Export review")
display(df_export_review)

zip_path = PIPE_DIR / "exports_datasets_full.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(EXPORT_DIR.glob("*.csv")):
        z.write(p, arcname=f"exports/{p.name}")

    zip_add_dir(z, FILTERED_DIR, "filtered")
    zip_add_dir(z, GOLD_DIR, "gold")
    zip_add_dir(z, NOTES_DIR, "notes")
    zip_add_dir(z, RECON_DIR, "reconstruction")
    zip_add_dir(z, SCORES_DIR, "scores")
    zip_add_dir(z, VERIFICATION_DIR, "verification")

print("ZIP saved:", zip_path)

# Optional: on Colab, trigger download dialog
try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
except Exception:
    pass


Index dataset (from filtered cache)


,patient_id,file_path,n_conditions,n_medication_requests
0,01d78eb5-7f50-45e9-f524-921196a3dffe,synthea_local_simplified/filtered_conditions_medreq/01d78eb5-7f50-45e9-f524-921196a3dffe.json,10,7
1,01f8bbfd-cfc6-3b97-8bc1-8da6f0b4a9a8,synthea_local_simplified/filtered_conditions_medreq/01f8bbfd-cfc6-3b97-8bc1-8da6f0b4a9a8.json,9,6
2,0288c42c-43a1-9878-4a9d-6b96caa12c40,synthea_local_simplified/filtered_conditions_medreq/0288c42c-43a1-9878-4a9d-6b96caa12c40.json,4,5
3,02b1604a-1bad-bc6f-5b72-d219ecd8f7f4,synthea_local_simplified/filtered_conditions_medreq/02b1604a-1bad-bc6f-5b72-d219ecd8f7f4.json,7,2
4,03777c32-ed98-50e2-f75d-cbcad532c610,synthea_local_simplified/filtered_conditions_medreq/03777c32-ed98-50e2-f75d-cbcad532c610.json,7,10
5,03a870d0-f88c-5cf7-538d-491923a3b57b,synthea_local_simplified/filtered_conditions_medreq/03a870d0-f88c-5cf7-538d-491923a3b57b.json,8,1
6,03b291e7-6d3f-d687-eac7-cb46455f34e3,synthea_local_simplified/filtered_conditions_medreq/03b291e7-6d3f-d687-eac7-cb46455f34e3.json,7,1
7,03bb882e-730f-0d96-cd3d-3caee932fef6,synthea_local_simplified/filtered_conditions_medreq/03bb882e-730f-0d96-cd3d-3caee932fef6.json,12,5
8,03e502b6-b810-06c1-7d65-83db077ed3ee,synthea_local_simplified/filtered_conditions_medreq/03e502b6-b810-06c1-7d65-83db077ed3ee.json,18,10
9,04f25f73-04b2-469c-3806-540417a0d61c,synthea_local_simplified/filtered_conditions_medreq/04f25f73-04b2-469c-3806-540417a0d61c.json,8,7


Generation dataset view (from cache) - gold source + generated notes


,patient_id,full_name,birthDate,gender,note_style,note_style_label,conditions_preview,medreq_preview,raw_note_preview,note_preview,raw_chars,clean_chars
0,03777c32-ed98-50e2-f75d-cbcad532c610,Eloise59 Gerhold939,1968-03-26,female,consult_note_short,Short consultation note,"Body mass index 30+ - obesity (finding); Localized, primary osteoarthritis of the hand; Acute bronchitis (disorder); Coronary Heart Disease; Severe anxiety (panic) (finding",Naproxen sodium 220 MG Oral Tablet; Acetaminophen 21.7 MG/ML / Dextromethorphan Hydrobromide 1 MG/ML / doxylamine succinate 0.417 MG/ML Oral Solution; Clopidogrel 75 MG Oral Tablet; Simvastatin 20 MG Oral Tablet; Amlodipine 5 MG Oral Tablet; Nitroglycerin 0.4 MG/ACTUAT Mucosal Spray,"Eloise59 Gerhold939 presented today complaining of localized, primary osteoarthritis of the hand and acute bronchitis. Given her Body mass index 30+ - obesity, I am concerned about her Coronary Heart Disease and initiated Clopidogrel 75 MG ...","Eloise59 Gerhold939 presented today complaining of localized, primary osteoarthritis of the hand and acute bronchitis. Given her Body mass index 30+ - obesity, I am concerned about her Coronary Heart Disease and initiated Clopidogrel 75 MG ...",1188,991
1,1ea40a05-c295-8b66-66a2-fda3bc1ed13f,Brock407 Paucek755,1960-05-25,male,consult_note_short,Short consultation note,Anemia (disorder); Acute bronchitis (disorder); Acute viral pharyngitis (disorder); Facial laceration,ferrous sulfate 325 MG Oral Tablet; Acetaminophen 325 MG Oral Tablet; Naproxen sodium 220 MG Oral Tablet; Amoxicillin 250 MG / Clavulanate 125 MG Oral Tablet,Brock407 Paucek755 presented today with a complaint of a sore throat and cough. He also has a facial laceration. Examination revealed signs of acute viral pharyngitis. I also noted that he is anemic. He is currently taking ferrous sulfate 3...,Brock407 Paucek755 presented today with a complaint of a sore throat and cough. He also has a facial laceration. Examination revealed signs of acute viral pharyngitis. I also noted that he is anemic. He is currently taking ferrous sulfate 3...,1162,612
2,51c7ff6a-33e3-3e1b-d3ad-0035c8227dfa,Georgiann138 Heaney114,2002-04-11,female,consult_note_short,Short consultation note,Streptococcal sore throat (disorder); Viral sinusitis (disorder); Chronic low back pain (finding); Chronic neck pain (finding); Hypertension,Ibuprofen 100 MG Oral Tablet; Penicillin V Potassium 250 MG Oral Tablet; Natazia 28 Day Pack; Acetaminophen 325 MG / HYDROcodone Bitartrate 7.5 MG Oral Tablet; Naproxen sodium 220 MG Oral Tablet; Levora 0.15/30 28 Day Pack,"Georgiann138 Heaney114 presented today. She is a female with a birthday of 2002-04-11. She is complaining of a streptococcal sore throat; the date of birth is April 11, 2002. She also has a history of chronic low back pain and chronic neck ...","Georgiann138 Heaney114 presented today. She is a female with a birthday of 2002-04-11. She is complaining of a streptococcal sore throat; the date of birth is April 11, 2002. She also has a history of chronic low back pain and chronic neck ...",1029,942
3,752a3c69-44d7-d6af-0340-7d3e748f6060,Oralia106 Williamson769,2000-08-10,female,consult_note_short,Short consultation note,Fracture of ankle,Naproxen sodium 220 MG Oral Tablet; Yaz 28 Day Pack; Natazia 28 Day Pack; Levora 0.15/30 28 Day Pack,"Oralia Williamson, a 22-year-old female patient, presented today complaining of ongoing ankle pain. She also mentioned experiencing some irregular bleeding. I have requested a Naproxen sodium 220 MG Oral Tablet for her ankle pain. She also ...","Oralia Williamson, a 22-year-old female patient, presented today complaining of ongoing ankle pain. She also mentioned experiencing some irregular bleeding. I have requested a Naproxen sodium 220 MG Oral Tablet for her ankle pain. She also ...",1309,420
4,7d28d76a-9ac8-67b4-3c88-0a75be3d0851,Erna640 Robel940,1969-01-03,female,consult_note_short,Short consultation note,Normal pregnancy; Anemia (disorder); Acute v

Note style distribution (from cache)


...


Tip: to inspect one patient in full, pick a patient_id and do:
  pid = df_gold_cache.iloc[0]['patient_id']
  df_gold_cache.loc[df_gold_cache.patient_id==pid, ['conditions','medication_requests']].iloc[0].to_dict()
  df_recon_cache.loc[df_recon_cache.patient_id==pid, ['raw_conditions','conditions','conditions_clipped']].iloc[0].to_dict()
  print(df_notes_cache.loc[df_notes_cache.patient_id==pid, 'note_text'].iloc[0])
Saved: data_bupa_conditions_medreq/exports/df_index.csv
Saved: data_bupa_conditions_medreq/exports/df_gold.csv
Saved: data_bupa_conditions_medreq/exports/df_notes.csv
Saved: data_bupa_conditions_medreq/exports/df_recon.csv
Saved: data_bupa_conditions_medreq/exports/df_scores.csv
Saved: data_bupa_conditions_medreq/exports/df_gold_conditions_long.csv
Saved: data_bupa_conditions_medreq/exports/df_gold_medreq_long.csv
Saved: data_bupa_conditions_medreq/exports/df_recon_conditions_long.csv
Saved: data_bupa_conditions_medreq/exports/df_recon_medreq_long.csv
Saved: data_bupa_cond

,dataset_name,rows,cols,csv_path
0,df_error_taxonomy,40,10,data_bupa_conditions_medreq/exports/df_error_taxonomy.csv
1,df_error_taxonomy_by_note_style_conditions,18,4,data_bupa_conditions_medreq/exports/df_error_taxonomy_by_note_style_conditions.csv
2,df_error_taxonomy_by_note_style_medication_requests,21,4,data_bupa_conditions_medreq/exports/df_error_taxonomy_by_note_style_medication_requests.csv
3,df_gold,40,15,data_bupa_conditions_medreq/exports/df_gold.csv
4,df_gold_conditions_long,216,3,data_bupa_conditions_medreq/exports/df_gold_conditions_long.csv
5,df_gold_medreq_long,212,3,data_bupa_conditions_medreq/exports/df_gold_medreq_long.csv
6,df_hard_cases_by_note_style,12,11,data_bupa_conditions_medreq/exports/df_hard_cases_by_note_style.csv
7,df_index,555,4,data_bupa_conditions_medreq/exports/df_index.csv
8,df_notes,40,14,data_bupa_conditions_medreq/exports/df_notes.csv
9,df_recon,40,36,data_bupa_conditions_medreq/exports/df_recon.csv


ZIP saved: data_bupa_conditions_medreq/exports_datasets_full.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>